# Boruta-Driven Explainable ML for PD LCM Transcriptomes — Merged Pipeline

The same end-to-end pipeline as **`pd-lcm-unified-pipeline`**, section for
section, run on every downloadable human laser-capture dataset of substantia
nigra dopamine neurons merged into one cohort. It writes the same CSV names, so
the companion notebook **`pd-lcm-merged-figures`** draws the same figures from it.

## Why merge

GSE182622's 192 samples are replicate pools from **22 donors**. A pool-level
train/test split puts pools of the same person on both sides, which inflates
every AUC. The unit of analysis here is therefore **the person**: GSE182622 pools
are summed per donor, and three more laser-capture datasets are added.

| Dataset | Platform | Control | PD | Unit |
|---|---|---|---|---|
| GSE182622 | RNA-seq | 10 | 12 | donor (pools summed) |
| GSE20141 | Affymetrix HG-U133 Plus 2 | 8 | 10 | person |
| GSE24378 | Affymetrix HG-U133 X3P | 9 | 8 | person |
| GSE169755 | RNA-seq | 3 | 3 | brain (2 replicates summed) |
| **total** | | **30** | **33** | **63 people** |

GSE114918 is excluded (diagnosis confounded with batch); E-MEXP-1416 duplicates
GSE24378. GSE7621 (bulk substantia nigra, 9 control / 16 PD) stays the external
cohort.

## Input datasets

| Source | File | Role |
|---|---|---|
| `alisaremi/count-parkinson` | `GSE182622_LCM_HS_Neuromelanin_Control_PD_ILBD_counts.txt` | discovery, dataset 1 |
| `alisaremi/externalvalidation2` | `GSE7621_series_matrix.txt` | external cohort |
| GEO (downloaded in §1) | GSE20141, GSE24378, GSE169755 series matrices; GSE169755 raw counts | discovery, datasets 2-4 |

Internet must be on: §1 downloads from GEO and maps identifiers with g:Profiler.

## What changed from the original pipeline, and why

| Step | Original | Merged |
|---|---|---|
| unit | LCM pool | person |
| matrix | raw counts of 8,000 HVGs | log expression, z-scored **within each dataset** |
| genes | 8,000 HVGs | genes measured on all four platforms and expressed in every dataset |
| split | stratified by diagnosis | stratified by dataset and diagnosis |
| DE | Wilcoxon on CP10K | Wilcoxon on the harmonised matrix, fold change from each dataset's own scale |
| Boruta | `perc=100`, `max_iter=50` | broadened (see §0) so the panel is about 20 genes |
| g:Profiler | exported file | queried live, same settings (custom background, Benjamini-Hochberg) |
| §14 | three reproduction quirks | leave-one-dataset-out, standard Boruta, nested CV |

Within-dataset standardisation removes platform differences but keeps each
person's position relative to the others in the same study. §1 checks that it
worked: after harmonisation a classifier should no longer tell which dataset a
person came from.

## 0. Configuration

Everything tunable lives here.

In [ ]:
# ============================== CONFIGURATION ==============================
RUN_ROBUSTNESS = True        # section 14 (leave-one-dataset-out etc., ~10 min)
RUN_LODO       = True        # leave-one-dataset-out inside section 14
RUN_NESTED_CV  = True        # nested-CV panel selection inside section 14

RANDOM_STATE = 42
TEST_SIZE    = 0.15          # held-out test fraction
VAL_SIZE     = 0.18          # validation fraction of the remaining data
N_SHAP_RUNS  = 10            # multi-run SHAP iterations
N_BOOT       = 2000          # bootstrap resamples for AUC CIs
PANEL_SIZE   = 5             # exhaustive search panel size

# ---- harmonisation --------------------------------------------------------
DATASETS  = ["GSE182622", "GSE20141", "GSE24378", "GSE169755"]
MIN_COUNT = 10               # GSE182622 pool filter: >= MIN_COUNT counts ...
MIN_FRAC  = 0.15             # ... in >= MIN_FRAC of Control+PD pools
EXPR_PCTL = 40               # a gene must reach this percentile of mean
                             # expression in EVERY dataset

# ---- Boruta, broadened ----------------------------------------------------
# perc=100 (the original) compares each gene with the BEST shadow feature. On 43
# training people that confirms only 8-10 genes. perc=99 compares with the 99th
# percentile of the shadows instead and keeps about 20; perc=98 already keeps 44
# and perc=90 keeps 118. Section 14B runs the standard setting side by side.
BORUTA_PERC     = 99
BORUTA_ALPHA    = 0.05
BORUTA_MAX_ITER = 100

# ---- differential expression ----------------------------------------------
DE_FDR, DE_LFC = 0.05, 1.0   # the original DEG rule
NOMINAL_P      = 0.01        # used only if the DEG rule returns < MIN_DEGS genes
MIN_DEGS       = 5

# ---- inputs ---------------------------------------------------------------
F_COUNTS  = "GSE182622_LCM_HS_Neuromelanin_Control_PD_ILBD_counts.txt"
F_GSE7621 = "GSE7621_series_matrix.txt"
GEO = "https://ftp.ncbi.nlm.nih.gov/geo/series"
GEO_FILES = {
    "GSE20141_series_matrix.txt.gz":  f"{GEO}/GSE20nnn/GSE20141/matrix/GSE20141_series_matrix.txt.gz",
    "GSE24378_series_matrix.txt.gz":  f"{GEO}/GSE24nnn/GSE24378/matrix/GSE24378_series_matrix.txt.gz",
    "GSE169755_series_matrix.txt.gz": f"{GEO}/GSE169nnn/GSE169755/matrix/GSE169755_series_matrix.txt.gz",
    "GSE169755_raw_counts.txt.gz":    f"{GEO}/GSE169nnn/GSE169755/suppl/GSE169755_raw_counts.txt.gz",
}
PROBE_NS = {"GSE20141": "AFFY_HG_U133_PLUS_2", "GSE24378": "AFFY_HG_U133_X3P"}

# ---- reference values from the local run of this notebook -----------------
EXPECT = {"n_people": 63, "n_control": 30, "n_pd": 33,
          "per_dataset": {"GSE182622": (10, 12), "GSE20141": (8, 10),
                          "GSE24378": (9, 8), "GSE169755": (3, 3)},
          "n_train": 43, "n_val": 10, "n_test": 10}
print("Merged pipeline | Boruta perc =", BORUTA_PERC, "| max_iter =", BORUTA_MAX_ITER)

In [ ]:
# ============================== INSTALL ==============================
# Kaggle's scipy/scikit-learn are compiled against the image's numpy. Installing
# shap/boruta unconstrained lets pip upgrade numpy, after which importing sklearn
# dies. So: only install what is missing, with the numeric stack pinned.
import importlib
import subprocess
import sys
from pathlib import Path

REQUIRED = [("shap", "shap"), ("boruta", "Boruta"),
            ("matplotlib_venn", "matplotlib-venn")]

missing = []
for module, package in REQUIRED:
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)

if missing:
    import numpy, scipy, sklearn, pandas
    pinned = {"numpy": numpy.__version__, "scipy": scipy.__version__,
              "scikit-learn": sklearn.__version__, "pandas": pandas.__version__}
    print("Pinning the existing numeric stack:")
    for k, v in pinned.items():
        print(f"  {k}=={v}")
    constraints = Path("/tmp/pd_constraints.txt")
    constraints.write_text("\n".join(f"{k}=={v}" for k, v in pinned.items()) + "\n")
    print(f"\nInstalling: {', '.join(missing)}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), *missing],
        capture_output=True, text=True)
    if result.returncode != 0:
        print("pip reported a problem:\n", result.stdout[-1500:], result.stderr[-1500:])
    import importlib.metadata as md
    for name, want in pinned.items():
        got = md.version(name)
        if got != want:
            raise RuntimeError(f"{name} moved {want} -> {got}; restart the session and re-run.")
    print("Numeric stack unchanged - safe to import sklearn.")
else:
    print("All dependencies already present; nothing installed.")

import numpy, scipy, sklearn, shap, boruta, matplotlib_venn
print("\nDependencies ready.")
print(f"  numpy {numpy.__version__} | scipy {scipy.__version__} | "
      f"sklearn {sklearn.__version__} | shap {shap.__version__}")

In [ ]:
# ============================== IMPORTS & SETUP ==============================
import os, re, io, gzip, json, time, glob, warnings, itertools, urllib.request
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib

try:
    get_ipython()
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_val_score,
                                     cross_val_predict)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report, roc_curve,
                             auc, roc_auc_score, f1_score, balanced_accuracy_score)

warnings.filterwarnings("ignore")

# Boruta needs the legacy numpy scalar aliases on numpy >= 1.24
for _alias, _target in [("float", float), ("int", int), ("bool", bool),
                        ("object", object), ("str", str)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)

import shap
from boruta import BorutaPy

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 11,
    "axes.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 110,
})

C_UP, C_DOWN = "#B2182B", "#2166AC"
C_BEST, C_OVERLAP, C_TOP5 = "#B2182B", "#D6604D", "#1B7837"
C_DEG5, C_DIST, C_CHANCE = "#2166AC", "#4575B4", "#BBBBBB"
TOP5_COLORS    = ["#1B7837", "#762A83", "#E08214", "#2166AC", "#B2182B"]
OVERLAP_COLORS = ["#4DAF4A", "#984EA3", "#FF7F00", "#A65628", "#F781BF",
                  "#999999", "#66C2A5", "#E78AC3"]
DS_COLORS = {"GSE182622": "#B62436", "GSE20141": "#2166AC",
             "GSE24378": "#E08E2B", "GSE169755": "#4D9A6A"}

ON_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR   = Path("/kaggle/working/outputs") if ON_KAGGLE else Path("./outputs")
FIG_DIR   = OUT_DIR / "figures"
GEO_CACHE = Path("/tmp/geo_cache") if ON_KAGGLE else Path("./geo_cache")
for _d in (OUT_DIR, FIG_DIR, GEO_CACHE):
    _d.mkdir(parents=True, exist_ok=True)

WRITTEN = []

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str

def show_table(df, caption=None, n=None):
    if caption:
        print(f"\n{caption}")
    view = df if n is None else df.head(n)
    if IN_NOTEBOOK:
        display(view)
    else:
        print(view.to_string(index=False))

def save_csv(df, name, description, preview=None):
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    WRITTEN.append({"file": name, "rows": len(df), "columns": df.shape[1],
                    "description": description})
    print(f"  [csv] {name}  ({len(df)} rows x {df.shape[1]} cols)")
    if preview:
        show_table(df, n=preview)
    return path

def save_fig(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=300, bbox_inches="tight")
    print(f"  [fig] {stem}.pdf / .png  (saved to outputs/figures/)")
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)

def find_input(filename):
    """Locate a file anywhere under /kaggle/input regardless of dataset slug."""
    for root in ["/kaggle/input", ".", ".."]:
        if not Path(root).exists():
            continue
        hits = glob.glob(f"{root}/**/{filename}", recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    raise FileNotFoundError(f"{filename!r} not found. Attach alisaremi/count-parkinson "
                            "and alisaremi/externalvalidation2.")

def fetch(filename):
    """An attached copy wins; otherwise download from GEO once and cache it."""
    try:
        return find_input(filename)
    except FileNotFoundError:
        pass
    dest = GEO_CACHE / filename
    if dest.exists() and dest.stat().st_size > 0:
        return str(dest)
    for k in range(5):
        try:
            print(f"  downloading {filename} from GEO ...")
            urllib.request.urlretrieve(GEO_FILES[filename], dest)
            return str(dest)
        except Exception as exc:
            print(f"    attempt {k + 1} failed: {type(exc).__name__}: {exc}")
            time.sleep(10 * (k + 1))
    raise RuntimeError(f"could not download {filename}; is Internet on?")

def gconvert(ids, target="ENSG"):
    """g:Profiler g:Convert. Returns {query: set(converted)} and {converted: name}."""
    out, names, ids = {}, {}, list(ids)
    for s in range(0, len(ids), 3000):
        body = {"organism": "hsapiens", "target": target, "query": ids[s:s + 3000]}
        res = None
        for k in range(6):
            try:
                req = urllib.request.Request(
                    "https://biit.cs.ut.ee/gprofiler/api/convert/convert/",
                    data=json.dumps(body).encode(),
                    headers={"Content-Type": "application/json"})
                with urllib.request.urlopen(req, timeout=300) as fh:
                    res = json.loads(fh.read())["result"]
                break
            except Exception as exc:
                print(f"    g:Convert retry {k + 1}: {type(exc).__name__}")
                time.sleep(8 * (k + 1))
        if res is None:
            raise RuntimeError("g:Profiler g:Convert unreachable; is Internet on?")
        for r in res:
            c = r.get("converted")
            if c and c not in ("None", "N/A"):
                out.setdefault(r["incoming"], set()).add(c)
                nm = r.get("name")
                if nm and nm not in ("None", "N/A", "nan"):
                    names.setdefault(c, nm)
    return out, names

def check(label, got, want, tol=None):
    ok = (abs(got - want) <= tol) if tol is not None else (got == want)
    print(f"  [{'OK ' if ok else 'DRIFT'}] {label}: {got}"
          + ("" if ok else f"   <-- expected {want}"))
    return ok

print(f"Output directory: {OUT_DIR.resolve()}")
print(f"Running on Kaggle: {ON_KAGGLE}")

## 1. Data loading and harmonisation

Each dataset is brought to **one row per person on its own log scale** first,
then the four are joined:

- **GSE182622** — Control and PD pools, the original low-expression filter, pools
  summed per donor (22 donors), log2 CPM.
- **GSE169755** — the two replicates of each brain summed, symbols mapped to
  Ensembl, log2 CPM.
- **GSE20141 / GSE24378** — the GEO series-matrix MAS5 values, log2, quantile
  normalised within the study, probes mapped to Ensembl, highest-mean probe kept.

Then: genes measured on all four platforms, expressed at or above the
`EXPR_PCTL` percentile in every dataset, **z-scored within each dataset** without
looking at diagnosis.

In [ ]:
# ============================== LOAD: GSE182622 (donor pseudobulk) ==============================
def logcpm(C):
    return np.log2(C / C.sum(0) * 1e6 + 1)

def series_matrix(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", errors="replace") as fh:
        lines = fh.read().split("\n")
    b = lines.index("!series_matrix_table_begin"); e = lines.index("!series_matrix_table_end")
    M = pd.read_csv(io.StringIO("\n".join(lines[b + 1:e])), sep="\t", index_col=0)
    meta = {}
    for l in lines:
        if l.startswith("!Sample_"):
            meta.setdefault(l.split("\t")[0], []).append(
                [x.strip('"') for x in l.split("\t")[1:]])
    return M, meta

counts_path = find_input(F_COUNTS)
counts_raw = pd.read_csv(counts_path, sep="\t", index_col=0)
print("GSE182622 raw matrix:", counts_raw.shape)

def parse_pool(s):
    p = s.split("-")
    return {"sample": s, "group": {"C": "Control", "PD": "PD", "I": "ILBD"}[p[1]],
            "donor": "-".join(p[1:4])}
pools = pd.DataFrame([parse_pool(s) for s in counts_raw.columns if s.startswith("DA-")])
pools = pools[pools["group"].isin(["Control", "PD"])].reset_index(drop=True)

clean = counts_raw[pools["sample"]].copy()
clean.index = clean.index.str.replace(r"\.\.\d+$", "", regex=True)
clean = clean.groupby(clean.index).sum()
keep = (clean >= MIN_COUNT).sum(axis=1) >= int(MIN_FRAC * clean.shape[1])
clean = clean.loc[keep]
PB = pd.DataFrame({d: clean[g["sample"].tolist()].sum(axis=1)
                   for d, g in pools.groupby("donor")})

LOG, PEOPLE = {}, []
LOG["GSE182622"] = logcpm(PB)
for d, g in pools.groupby("donor"):
    PEOPLE.append({"dataset": "GSE182622", "person": d, "y": int(g["group"].iloc[0] == "PD"),
                   "units_merged": len(g), "platform": "RNA-seq (LCM pools)"})
print(f"  {len(pools)} pools -> {PB.shape[1]} donors; {PB.shape[0]:,} genes after the pool filter")
pools_per_donor = pools.groupby(["donor", "group"]).size().rename("pools").reset_index()

In [ ]:
# ============================== LOAD: GSE169755 and the two microarray studies ==============================
# ---- GSE169755: two replicates per brain ------------------------------------
C2 = pd.read_csv(fetch("GSE169755_raw_counts.txt.gz"), sep="\t", index_col=0)
_, m2 = series_matrix(fetch("GSE169755_series_matrix.txt.gz"))
grp_row = next(r for r in m2["!Sample_characteristics_ch1"] if any("group:" in x for x in r))
brain_group = {}
for t, g in zip(m2["!Sample_title"][0], grp_row):
    brain_group[re.search(r"Brain (\d+)", t).group(1)] = g.split(": ", 1)[1]
PB2 = pd.DataFrame({f"Brain{b}": C2[[c for c in C2.columns if c[:-1] == b]].sum(axis=1)
                    for b in brain_group})
sym2ens, _ = gconvert(PB2.index.tolist(), "ENSG")
PB2 = PB2.loc[[s for s in PB2.index if s in sym2ens and len(sym2ens[s]) == 1]]
PB2.index = [next(iter(sym2ens[s])) for s in PB2.index]
PB2 = PB2.groupby(level=0).sum()
LOG["GSE169755"] = logcpm(PB2)
for b, gname in brain_group.items():
    PEOPLE.append({"dataset": "GSE169755", "person": f"Brain{b}", "y": int("Parkinson" in gname),
                   "units_merged": 2, "platform": "RNA-seq (LCM)"})
print(f"GSE169755: {C2.shape[1]} libraries -> {PB2.shape[1]} brains, {PB2.shape[0]:,} genes")

# ---- microarrays: log2 MAS5, quantile-normalise, probe -> gene ---------------
def quantile_norm(M):
    r = M.rank(axis=0, method="first").astype(int) - 1
    ref = np.sort(M.to_numpy(), axis=0).mean(1)
    return pd.DataFrame(ref[r.to_numpy()], index=M.index, columns=M.columns)

def collapse_probes(L, pmap):
    means, best = L.mean(1), {}
    for p, gs in pmap.items():
        if p not in L.index or len(gs) != 1:
            continue
        g = next(iter(gs))
        if g not in best or means[p] > means[best[g]]:
            best[g] = p
    return pd.DataFrame({g: L.loc[p] for g, p in best.items()}).T

for acc in ("GSE20141", "GSE24378"):
    M, m = series_matrix(fetch(f"{acc}_series_matrix.txt.gz"))
    L = quantile_norm(np.log2(M.clip(lower=1)))
    pmap, _ = gconvert(L.index.tolist(), "ENSG")
    G = collapse_probes(L, pmap)
    titles = m["!Sample_title"][0]
    dis = next(v for v in m["!Sample_characteristics_ch1"] if any("disease state" in x for x in v))
    G.columns = titles
    LOG[acc] = G
    for t, dname in zip(titles, dis):
        PEOPLE.append({"dataset": acc, "person": t, "y": int("Parkinson" in dname),
                       "units_merged": 1, "platform": f"microarray ({PROBE_NS[acc]})"})
    print(f"{acc}: {M.shape[0]:,} probes x {M.shape[1]} arrays -> {G.shape[0]:,} genes")

PEOPLE = pd.DataFrame(PEOPLE)
cohort = (PEOPLE.groupby("dataset")
          .agg(platform=("platform", "first"), control=("y", lambda s: int((s == 0).sum())),
               PD=("y", "sum"), people=("y", "size"), units_merged=("units_merged", "sum"))
          .loc[DATASETS].reset_index())
save_csv(cohort, "01_cohort_by_dataset.csv", "People per dataset and diagnosis after merging units")
show_table(cohort, "Cohort:")

print("\nReproduction checks:")
for d, (nc, npd) in EXPECT["per_dataset"].items():
    r = cohort.set_index("dataset").loc[d]
    check(f"{d} control/PD", (int(r["control"]), int(r["PD"])), (nc, npd))

In [ ]:
# ============================== HARMONISE ==============================
common = set.intersection(*[set(LOG[d].index) for d in DATASETS])
thr = {d: np.percentile(LOG[d].mean(1), EXPR_PCTL) for d in DATASETS}
means = {d: LOG[d].mean(1) for d in DATASETS}
GENES = sorted(g for g in common if all(means[d][g] >= thr[d] for d in DATASETS))

LOGM, Z, META = [], [], []
for d in DATASETS:
    Md = LOG[d].loc[GENES]
    ppl = PEOPLE[PEOPLE["dataset"] == d].set_index("person").loc[Md.columns]
    LOGM.append(Md.T)
    Z.append(Md.sub(Md.mean(1), axis=0).div(Md.std(1, ddof=1).clip(lower=0.05), axis=0).T)
    META.append(pd.DataFrame({"dataset": d, "person": Md.columns, "y": ppl["y"].to_numpy()}))
LOGM = pd.concat(LOGM)
XZ   = pd.concat(Z)
META = pd.concat(META).reset_index(drop=True)

X          = XZ.to_numpy(np.float32)
y          = META["y"].to_numpy()
DS         = META["dataset"].to_numpy()
gene_names = list(GENES)
STRATA     = np.array([f"{d}_{v}" for d, v in zip(DS, y)])

_, SYMBOL = gconvert(GENES, "ENSG")
def sym(g):
    return SYMBOL.get(g, g)

print(f"genes on every platform: {len(common):,}")
print(f"expressed in every dataset (>= {EXPR_PCTL}th percentile): {len(GENES):,}")
print(f"merged matrix: {X.shape[0]} people x {X.shape[1]:,} genes "
      f"(Control={int((y == 0).sum())}, PD={int(y.sum())})")
print(f"gene symbols resolved: {sum(g in SYMBOL for g in GENES):,}/{len(GENES):,}")

prep = pd.DataFrame([
    {"step": "GSE182622 genes after pool filter", "genes": LOG["GSE182622"].shape[0], "samples": PB.shape[1]},
    {"step": "GSE169755 genes",                  "genes": LOG["GSE169755"].shape[0], "samples": PB2.shape[1]},
    {"step": "GSE20141 genes",                   "genes": LOG["GSE20141"].shape[0], "samples": LOG["GSE20141"].shape[1]},
    {"step": "GSE24378 genes",                   "genes": LOG["GSE24378"].shape[0], "samples": LOG["GSE24378"].shape[1]},
    {"step": "raw matrix",                       "genes": int(pd.concat([LOG[d] for d in DATASETS], axis=1).shape[0]), "samples": len(META)},
    {"step": "genes on every platform",          "genes": len(common),  "samples": len(META)},
    {"step": "expressed in every dataset (model X)", "genes": len(GENES), "samples": len(META)},
])
save_csv(prep, "01_preprocessing_summary.csv", "Gene/person counts at each harmonisation step")
save_csv(pools_per_donor, "01_gse182622_pools_per_donor.csv", "GSE182622 pools summed into each donor")

print("\nReproduction checks:")
check("people", X.shape[0], EXPECT["n_people"])
check("controls", int((y == 0).sum()), EXPECT["n_control"])
check("PD", int(y.sum()), EXPECT["n_pd"])

In [ ]:
# ============================== DATASET-IDENTITY CHECK ==============================
# If harmonisation worked, a classifier should no longer be able to tell which
# dataset a person came from. Before: each dataset's log scale, centred once.
def dataset_predictability(Xm, labels, reps=10):
    accs = []
    for r in range(reps):
        cv = StratifiedKFold(5, shuffle=True, random_state=r)
        p = cross_val_predict(LogisticRegression(max_iter=3000, C=0.1), Xm, labels, cv=cv)
        accs.append(balanced_accuracy_score(labels, p))
    return float(np.mean(accs))

X_before = LOGM.to_numpy(float)
X_before = X_before - X_before.mean(0)
acc_before = dataset_predictability(X_before, DS)
acc_after  = dataset_predictability(X, DS)
null = []
for k in range(50):
    perm = np.random.default_rng(k).permutation(DS)
    cv = StratifiedKFold(5, shuffle=True, random_state=k)
    p = cross_val_predict(LogisticRegression(max_iter=3000, C=0.1), X, perm, cv=cv)
    null.append(balanced_accuracy_score(perm, p))
null = np.array(null)
chance = 1 / len(DATASETS)
print(f"Can a model tell which dataset a person came from? (balanced accuracy, chance {chance:.2f})")
print(f"  before harmonisation: {acc_before:.2f}")
print(f"  after harmonisation : {acc_after:.2f}   (shuffled null mean {null.mean():.2f}, "
      f"95th percentile {np.percentile(null, 95):.2f})")
passed = acc_after <= np.percentile(null, 95)
print("  PASS - dataset identity is no longer recoverable" if passed else
      "  WARNING - dataset identity is still recoverable after harmonisation")

save_csv(pd.DataFrame([
    ("chance", chance), ("accuracy_before", acc_before), ("accuracy_after", acc_after),
    ("null_mean", float(null.mean())), ("null_95th", float(np.percentile(null, 95))),
    ("passed", int(passed)),
], columns=["statistic", "value"]), "01_dataset_identity_check.csv",
   "Balanced accuracy of predicting the dataset before/after harmonisation")

pca_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, Xm, stage, acc in ((axes[0], X_before, "before", acc_before), (axes[1], X, "after", acc_after)):
    pc = PCA(2, random_state=0).fit(Xm); Zp = pc.transform(Xm)
    for i in range(len(META)):
        pca_rows.append({"stage": stage, "dataset": DS[i], "person": META["person"][i], "y": int(y[i]),
                         "PC1": Zp[i, 0], "PC2": Zp[i, 1],
                         "var_PC1": pc.explained_variance_ratio_[0], "var_PC2": pc.explained_variance_ratio_[1]})
    for d in DATASETS:
        for yy, mk in ((0, "o"), (1, "^")):
            m = (DS == d) & (y == yy)
            ax.scatter(Zp[m, 0], Zp[m, 1], c=DS_COLORS[d], marker=mk, s=46,
                       edgecolors="white", linewidths=0.6, label=f"{d} {'PD' if yy else 'control'}")
    ax.set_title(f"{stage.capitalize()} harmonisation\nmodel identifies dataset: {acc:.0%} "
                 f"(chance {chance:.0%})", fontsize=11)
    ax.set_xlabel(f"PC1 ({pc.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pc.explained_variance_ratio_[1]:.0%})")
axes[1].legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
save_fig(fig, "fig01_harmonisation_pca")
save_csv(pd.DataFrame(pca_rows), "01_pca_coordinates.csv", "PCA coordinates before and after harmonisation")

## 2. Baseline Random Forest on all harmonised genes

Stratified 3-way split, stratified by **dataset and diagnosis** so every dataset
is represented in each part, and the original baseline classifier.

In [ ]:
# ============================== SPLIT + BASELINE RF ==============================
idx_all = np.arange(len(y))
idx_train_full, idx_test = train_test_split(
    idx_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=STRATA)
idx_train, idx_val = train_test_split(
    idx_train_full, test_size=VAL_SIZE, random_state=RANDOM_STATE,
    stratify=STRATA[idx_train_full])

X_train_full, X_test = X[idx_train_full], X[idx_test]
y_train_full, y_test = y[idx_train_full], y[idx_test]
X_train, X_val = X[idx_train], X[idx_val]
y_train, y_val = y[idx_train], y[idx_val]

print(f"Train {len(idx_train)} | Val {len(idx_val)} | Test {len(idx_test)}")
split_df = META.assign(split="train")
split_df.loc[idx_val, "split"] = "val"
split_df.loc[idx_test, "split"] = "test"
show_table(pd.crosstab([split_df["dataset"], split_df["y"].map({0: "Control", 1: "PD"})],
                       split_df["split"]).reset_index(), "People per split:")

rf_model = RandomForestClassifier(
    n_estimators=800, max_depth=10, min_samples_split=4, min_samples_leaf=2,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)

test_pred = rf_model.predict(X_test)
test_prob = rf_model.predict_proba(X_test)[:, 1]
test_acc  = accuracy_score(y_test, test_pred)
print(f"\nTest accuracy: {test_acc:.4f}")
print(classification_report(y_test, test_pred, target_names=["Control", "PD"], zero_division=0))

rep = classification_report(y_test, test_pred, target_names=["Control", "PD"],
                            output_dict=True, zero_division=0)
baseline = pd.DataFrame([
    {"metric": "accuracy",  "class": "overall", "value": test_acc},
    {"metric": "roc_auc",   "class": "overall", "value": roc_auc_score(y_test, test_prob)},
    {"metric": "f1_macro",  "class": "overall", "value": rep["macro avg"]["f1-score"]},
    {"metric": "precision", "class": "Control", "value": rep["Control"]["precision"]},
    {"metric": "recall",    "class": "Control", "value": rep["Control"]["recall"]},
    {"metric": "f1",        "class": "Control", "value": rep["Control"]["f1-score"]},
    {"metric": "precision", "class": "PD",      "value": rep["PD"]["precision"]},
    {"metric": "recall",    "class": "PD",      "value": rep["PD"]["recall"]},
    {"metric": "f1",        "class": "PD",      "value": rep["PD"]["f1-score"]},
    {"metric": "n_train",   "class": "overall", "value": len(idx_train)},
    {"metric": "n_val",     "class": "overall", "value": len(idx_val)},
    {"metric": "n_test",    "class": "overall", "value": len(idx_test)},
])
save_csv(baseline, "02_baseline_rf_performance.csv",
         "Baseline RF on all harmonised genes: accuracy, AUC, per-class precision/recall/F1")
save_csv(split_df, "02_split_assignment.csv", "Which people are in train / val / test")

print("\nReproduction checks:")
check("train size", len(idx_train), EXPECT["n_train"])
check("val size",   len(idx_val),   EXPECT["n_val"])
check("test size",  len(idx_test),  EXPECT["n_test"])

## 3. Differential expression (PD vs Control)

Wilcoxon rank-sum on the harmonised matrix, as in the original, with the fold
change taken from each dataset's own log scale and averaged by size. A
random-effects meta-analysis (Hedges' *g* per dataset, DerSimonian-Laird) is
reported beside it so heterogeneity between datasets is visible.

The original rule is FDR < 0.05 and |log2FC| > 1. If fewer than `MIN_DEGS` genes
pass it, the Boruta ∩ DE comparison uses **nominal** *p* < `NOMINAL_P` instead,
and every output says so.

In [ ]:
# ============================== DIFFERENTIAL EXPRESSION ==============================
def bh(p):
    return stats.false_discovery_control(np.asarray(p), method="bh")

def wilcoxon_de(idx):
    Xi, yi, di = X[idx], y[idx], DS[idx]
    a, b = Xi[yi == 1], Xi[yi == 0]
    w = stats.mannwhitneyu(a, b, axis=0, alternative="two-sided")
    n1, n0 = len(a), len(b)
    score = (w.statistic - n1 * n0 / 2) / np.sqrt(n1 * n0 * (n1 + n0 + 1) / 12)
    Li = LOGM.to_numpy(float)[idx]
    lfc = np.zeros(X.shape[1]); ntot = 0
    for d in DATASETS:
        m = di == d
        if (m & (yi == 1)).any() and (m & (yi == 0)).any():
            lfc += m.sum() * (Li[m & (yi == 1)].mean(0) - Li[m & (yi == 0)].mean(0))
            ntot += m.sum()
    return score, lfc / max(ntot, 1), w.pvalue, bh(w.pvalue)

def meta_analysis(idx):
    Li, yi, di = LOGM.to_numpy(float)[idx], y[idx], DS[idx]
    G_, V_ = [], []
    for d in DATASETS:
        m = di == d
        a, b = Li[m & (yi == 1)], Li[m & (yi == 0)]
        if len(a) < 2 or len(b) < 2:
            continue
        n1, n0 = len(a), len(b)
        sp = np.sqrt(((n1 - 1) * a.var(0, ddof=1) + (n0 - 1) * b.var(0, ddof=1)) / (n1 + n0 - 2)).clip(min=1e-6)
        J = 1 - 3 / (4 * (n1 + n0) - 9)
        g = J * (a.mean(0) - b.mean(0)) / sp
        G_.append(g); V_.append((n1 + n0) / (n1 * n0) + g ** 2 / (2 * (n1 + n0)))
    G_, V_ = np.array(G_), np.array(V_); w = 1 / V_; k = len(G_)
    fixed = (w * G_).sum(0) / w.sum(0)
    Q = (w * (G_ - fixed) ** 2).sum(0)
    tau2 = np.maximum(0, (Q - (k - 1)) / (w.sum(0) - (w ** 2).sum(0) / w.sum(0)))
    wr = 1 / (V_ + tau2)
    gr = (wr * G_).sum(0) / wr.sum(0)
    zr = gr / np.sqrt(1 / wr.sum(0))
    pr = 2 * stats.norm.sf(np.abs(zr))
    I2 = np.where(Q > 0, np.maximum(0, (Q - (k - 1)) / np.maximum(Q, 1e-12)), 0)
    return gr, pr, I2, (np.sign(G_) == np.sign(gr)).sum(0)

score, lfc_all, p_all, padj_all = wilcoxon_de(idx_all)
g_meta, p_meta, I2_meta, same_dir = meta_analysis(idx_all)
de_results = pd.DataFrame({
    "gene": GENES, "symbol": [sym(g) for g in GENES], "scores": score, "log2FC": lfc_all,
    "pvalue": p_all, "padj": padj_all, "hedges_g_meta": g_meta, "meta_pvalue": p_meta,
    "meta_padj": bh(p_meta), "I2": I2_meta, "same_direction_datasets": same_dir,
}).sort_values(["padj", "pvalue"]).reset_index(drop=True)

degs_strict = de_results[(de_results["padj"] < DE_FDR) & (de_results["log2FC"].abs() > DE_LFC)]
print(f"FDR < {DE_FDR} and |log2FC| > {DE_LFC}: {len(degs_strict)} genes")
print(f"FDR < {DE_FDR} (any fold change): {int((de_results['padj'] < DE_FDR).sum())}")
print(f"meta-analysis FDR < {DE_FDR}: {int((de_results['meta_padj'] < DE_FDR).sum())}")
print(f"nominal p < {NOMINAL_P}: {int((de_results['pvalue'] < NOMINAL_P).sum())} "
      f"(about {NOMINAL_P * len(GENES):.0f} expected by chance alone)")

if len(degs_strict) >= MIN_DEGS:
    DE_RULE, DE_LABEL = "strict", f"DEGs (FDR < {DE_FDR}, |log2FC| > {DE_LFC})"
    degs = degs_strict.copy()
else:
    DE_RULE, DE_LABEL = "nominal", f"nominal DE genes (p < {NOMINAL_P}, not FDR-corrected)"
    degs = de_results[de_results["pvalue"] < NOMINAL_P].copy()
    print(f"\nNOTE: fewer than {MIN_DEGS} genes pass the original rule, so the Boruta ∩ DE")
    print(f"      comparison below uses {DE_LABEL}. None of them is FDR-significant.")
degs = degs.sort_values("pvalue").reset_index(drop=True)
de_results["is_deg"] = de_results["gene"].isin(set(degs["gene"]))
de_results["is_fdr_deg"] = de_results["gene"].isin(set(degs_strict["gene"]))

n_up, n_down = int((degs["log2FC"] > 0).sum()), int((degs["log2FC"] < 0).sum())
print(f"\n{DE_LABEL}: {len(degs)}  (up in PD: {n_up}, down in PD: {n_down})")

save_csv(de_results, "03_de_results_full.csv",
         "Wilcoxon DE (harmonised) with meta-analysis columns, all genes")
save_csv(degs, "03_degs_significant.csv", f"Genes in the DE set used downstream: {DE_LABEL}")
table1 = (de_results.sort_values("pvalue").head(20)[["gene", "symbol", "log2FC", "pvalue", "padj",
                                                     "hedges_g_meta", "I2", "same_direction_datasets"]]
          .assign(rank=lambda d: range(1, len(d) + 1)))
table1 = table1[["rank"] + [c for c in table1.columns if c != "rank"]]
save_csv(table1, "03_table1_top20_degs.csv", "Table 1: 20 most significant genes by p-value")
save_csv(pd.DataFrame([
    ("de_rule", DE_RULE), ("de_label", DE_LABEL), ("n_de_set", len(degs)),
    ("de_fdr", DE_FDR), ("de_lfc", DE_LFC), ("nominal_p", NOMINAL_P),
    ("n_fdr05_lfc1", len(degs_strict)), ("n_fdr05", int((de_results["padj"] < DE_FDR).sum())),
    ("n_meta_fdr05", int((de_results["meta_padj"] < DE_FDR).sum())),
    ("n_nominal_p", int((de_results["pvalue"] < NOMINAL_P).sum())),
    ("expected_nominal_by_chance", NOMINAL_P * len(GENES)), ("n_genes_tested", len(GENES)),
], columns=["statistic", "value"]), "03_de_settings.csv", "Which DE rule fed the downstream sets, and why")
show_table(table1, "Table 1 - 20 most significant genes:", n=10)

## 4. Boruta all-relevant feature selection

Same estimator as the original (balanced Random Forest, `n_estimators='auto'`,
`two_step=True`), fitted on the training people only. `perc` and `max_iter` come
from §0: the standard `perc=100` keeps 8-10 genes on 43 people, so it is lowered
to `perc=99`, which keeps about as many genes as the original panel (perc=98
already keeps 44). §14B runs the standard
setting beside it.

In [ ]:
# ============================== BORUTA ==============================
boruta_rf = RandomForestClassifier(
    n_estimators=500, max_depth=8, min_samples_split=4, min_samples_leaf=2,
    class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)

feat_selector = BorutaPy(
    estimator=boruta_rf, n_estimators="auto", perc=BORUTA_PERC, alpha=BORUTA_ALPHA,
    two_step=True, max_iter=BORUTA_MAX_ITER, random_state=RANDOM_STATE, verbose=0)
t0 = time.time()
feat_selector.fit(X_train, y_train)
print(f"Boruta finished in {time.time() - t0:.0f} s")

selected_mask  = feat_selector.support_
selected_genes = np.array(gene_names)[selected_mask]
X_train_boruta = X_train[:, selected_mask]
X_val_boruta   = X_val[:, selected_mask]
X_test_boruta  = X_test[:, selected_mask]

print(f"\nConfirmed relevant genes: {selected_mask.sum()}  "
      f"(tentative, not used: {int(feat_selector.support_weak_.sum())})")
print(", ".join(sym(g) for g in selected_genes))

boruta_df = pd.DataFrame({
    "gene": selected_genes, "symbol": [sym(g) for g in selected_genes],
    "boruta_rank": feat_selector.ranking_[selected_mask], "confirmed": True,
})
save_csv(boruta_df, "04_boruta_selected_genes.csv", "Boruta-confirmed relevant genes with ranking")
save_csv(pd.DataFrame([
    ("perc", BORUTA_PERC), ("alpha", BORUTA_ALPHA), ("max_iter", BORUTA_MAX_ITER),
    ("n_train", len(idx_train)), ("n_features", X_train.shape[1]),
    ("n_confirmed", int(selected_mask.sum())), ("n_tentative", int(feat_selector.support_weak_.sum())),
], columns=["setting", "value"]), "04_boruta_settings.csv", "Boruta settings and counts")

## 5. Multi-run SHAP

Ten independent Random Forest fits on the Boruta feature set, each explained with
`TreeExplainer` on the held-out test people. Averaging across runs gives
directional importance plus a stability estimate.

In [ ]:
# ============================== MULTI-RUN SHAP ==============================
shap_values_list, pred_list = [], []
for i in range(N_SHAP_RUNS):
    seed  = RANDOM_STATE + i * 11
    model = RandomForestClassifier(
        n_estimators=800, max_depth=10, min_samples_split=4, min_samples_leaf=2,
        class_weight="balanced", random_state=seed, n_jobs=-1)
    model.fit(X_train_boruta, y_train)
    vals = shap.TreeExplainer(model).shap_values(X_test_boruta)
    if isinstance(vals, list):
        vals = vals[1]
    elif getattr(vals, "ndim", 2) == 3:
        vals = vals[:, :, 1]
    shap_values_list.append(vals)
    pred_list.append(model.predict_proba(X_test_boruta)[:, 1])
print(f"{N_SHAP_RUNS} SHAP runs done")

mean_shap = np.mean(shap_values_list, axis=0)
std_shap  = np.std(shap_values_list, axis=0)
mean_abs_shap = np.abs(mean_shap).mean(axis=0)
mean_shap_g   = mean_shap.mean(axis=0)
shap_std_g    = std_shap.mean(axis=0)

importance_df = (pd.DataFrame({
        "gene": selected_genes, "symbol": [sym(g) for g in selected_genes],
        "mean_abs_shap": mean_abs_shap, "mean_shap": mean_shap_g,
        "shap_std": shap_std_g, "shap_cv": shap_std_g / (np.abs(mean_shap_g) + 1e-8)})
    .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
# Direction = does HIGHER expression push the call towards PD? Read from the
# correlation between each gene's expression and its SHAP value across the test
# people. The mean signed SHAP over a small, unbalanced test set is not a
# direction: it mostly reflects how many PD and control people happen to be in it.
_dir = []
for g in importance_df["gene"]:
    j = list(selected_genes).index(g)
    rho_ = stats.spearmanr(X_test_boruta[:, j], mean_shap[:, j])[0]
    _dir.append(np.sign(rho_) if np.isfinite(rho_) and rho_ != 0 else
                np.sign(de_results.set_index("gene").loc[g, "log2FC"]))
importance_df["direction_sign"] = np.array(_dir, dtype=int)
importance_df["signed_importance"] = importance_df["direction_sign"] * importance_df["mean_abs_shap"]
importance_df["direction"] = np.where(importance_df["direction_sign"] > 0,
                                      "Higher expression raises PD probability",
                                      "Higher expression lowers PD probability")
importance_df["shap_rank"] = range(1, len(importance_df) + 1)
save_csv(importance_df, "05_boruta_shap_importance.csv",
         f"Mean |SHAP|, directional SHAP and stability over {N_SHAP_RUNS} RF runs")
show_table(importance_df[["shap_rank", "symbol", "mean_abs_shap", "shap_std", "direction"]],
           "Boruta genes ranked by mean |SHAP|:")
top5_shap_genes = importance_df["gene"].astype(str).head(5).tolist()

In [ ]:
# ---------- Figure: directional SHAP bar plot + beeswarm ----------
import matplotlib.patches as mpatches
plot_df = importance_df.sort_values("mean_abs_shap", ascending=True).reset_index(drop=True)
colors  = [C_UP if v > 0 else C_DOWN for v in plot_df["signed_importance"]]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(np.arange(len(plot_df)), plot_df["signed_importance"], color=colors, height=0.62,
        edgecolor="white", linewidth=0.4)
ax.errorbar(plot_df["signed_importance"], np.arange(len(plot_df)), xerr=plot_df["shap_std"],
            fmt="none", ecolor="#555555", elinewidth=0.8, capsize=2.5, capthick=0.8, zorder=5)
ax.axvline(0, color="#333333", lw=0.9, zorder=3)
ax.set_yticks(np.arange(len(plot_df)))
ax.set_yticklabels(plot_df["symbol"], fontsize=8.5)
ax.set_xlabel("Mean |SHAP|, signed by whether higher expression raises PD probability")
ax.set_title(f"Directional SHAP importance\n(mean of {N_SHAP_RUNS} Random Forest runs)",
             fontweight="bold", pad=10)
ax.legend(handles=[mpatches.Patch(color=C_UP, label="higher expression -> PD"),
                   mpatches.Patch(color=C_DOWN, label="higher expression -> control")],
          loc="lower right", fontsize=9)
fig.tight_layout()
save_fig(fig, "fig02_shap_directional")

order = importance_df.sort_values("mean_abs_shap", ascending=True)["gene"].tolist()
idx   = [list(selected_genes).index(g) for g in order]
fig2 = plt.figure(figsize=(10, 7))
shap.summary_plot(mean_shap[:, idx], X_test_boruta[:, idx],
                  feature_names=np.array([sym(g) for g in order]), max_display=len(order),
                  show=False, plot_size=None)
plt.title(f"SHAP summary - PD vs Control\n(mean of {N_SHAP_RUNS} runs, Boruta genes)",
          fontweight="bold", pad=14)
plt.tight_layout()
save_fig(fig2, "fig03_shap_beeswarm")

## 6. Boruta ∩ DE overlap

The overlap panel is the intersection of the two discovery tracks.

In [ ]:
# ============================== OVERLAP ANALYSIS ==============================
boruta_set = set(map(str, selected_genes))
deg_set    = set(degs["gene"].astype(str))
overlap    = boruta_set & deg_set
overlap_genes = sorted(overlap, key=lambda g: de_results.set_index("gene").loc[g, "pvalue"])
print(f"Boruta: {len(boruta_set)} | {DE_LABEL}: {len(deg_set)} | overlap: {len(overlap)}")
print("Overlap genes:", [sym(g) for g in overlap_genes])

save_csv(pd.DataFrame([
    {"set": "Boruta only",  "n": len(boruta_set - deg_set)},
    {"set": "DEG only",     "n": len(deg_set - boruta_set)},
    {"set": "Overlap",      "n": len(overlap)},
    {"set": "Boruta total", "n": len(boruta_set)},
    {"set": "DEG total",    "n": len(deg_set)},
]), "06_overlap_analysis.csv", f"Venn counts: Boruta vs {DE_LABEL}")

de_lookup = de_results.set_index(de_results["gene"].astype(str))
all_genes = sorted(boruta_set | deg_set)
membership = pd.DataFrame({"gene": all_genes})
membership["symbol"]     = membership["gene"].map(sym)
membership["in_boruta"]  = membership["gene"].isin(boruta_set)
membership["in_deg"]     = membership["gene"].isin(deg_set)
membership["in_overlap"] = membership["gene"].isin(overlap)
membership["log2FC"]     = membership["gene"].map(de_lookup["log2FC"])
membership["pvalue"]     = membership["gene"].map(de_lookup["pvalue"])
membership["padj"]       = membership["gene"].map(de_lookup["padj"])
_imp = importance_df.set_index(importance_df["gene"].astype(str))
membership["mean_abs_shap"] = membership["gene"].map(_imp["mean_abs_shap"])
membership["shap_rank"]     = membership["gene"].map(_imp["shap_rank"])
save_csv(membership, "06_gene_set_membership.csv",
         "Per-gene membership in Boruta / DE / overlap sets with effect sizes")

In [ ]:
# ---------- Figure: Venn + volcano ----------
from matplotlib_venn import venn2
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
v = venn2([boruta_set, deg_set], set_labels=("Boruta", "DE set"), ax=axes[0])
for patch, colr in zip(v.patches, [C_TOP5, C_DEG5, C_OVERLAP]):
    if patch:
        patch.set_alpha(0.65); patch.set_color(colr)
axes[0].set_title(f"A  Boruta-selected vs {DE_LABEL}", fontweight="bold", loc="left", fontsize=10)

vol = de_results.copy()
vol["neglog10p"] = -np.log10(vol["pvalue"].clip(lower=1e-300))
ax = axes[1]
ax.scatter(vol["log2FC"], vol["neglog10p"], s=6, c="#DDDDDD", edgecolors="none", label="all genes")
sig = vol[vol["is_deg"]]
ax.scatter(sig["log2FC"], sig["neglog10p"], s=10, c=C_DEG5, edgecolors="none", label="DE set")
bor = vol[vol["gene"].astype(str).isin(boruta_set)]
ax.scatter(bor["log2FC"], bor["neglog10p"], s=45, c=C_UP, edgecolors="black", linewidths=0.5,
           label="Boruta", zorder=5)
ax.axhline(-np.log10(NOMINAL_P), ls="--", lw=0.8, color="#888888")
ax.set_xlabel("log2 fold change (PD vs Control)")
ax.set_ylabel("-log10 p-value")
ax.set_title("B  Boruta genes on the volcano landscape", fontweight="bold", loc="left")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "fig04_venn_volcano")

## 7. ROC evaluation of gene panels

Shared helpers, as in the original:

- `fit_lr_panel` — standardise on train, L2 logistic regression, probabilities on test.
- `single_gene_roc` — one gene, with **direction correction** (a curve below
  chance is flipped).
- `bootstrap_auc_ci` — non-parametric 95% CI, 2,000 resamples, seeded.

The split is the two-way train+val / test split; the test people are the same
ones as in §2.

> With about 10 test people an AUC moves in steps of roughly 0.04, and one
> person can shift it by more than that. §14 gives the leave-one-dataset-out
> numbers, which use every person.

In [ ]:
# ============================== ROC HELPERS ==============================
gene_names_list = list(map(str, gene_names))
_col = {g: i for i, g in enumerate(gene_names_list)}
X_tr_full, X_te, y_tr_full, y_te = X_train_full, X_test, y_train_full, y_test
print(f"X_train_full : {X_tr_full.shape}  (PD={int(y_tr_full.sum())}, Ctrl={int((y_tr_full == 0).sum())})")
print(f"X_test       : {X_te.shape}  (PD={int(y_te.sum())}, Ctrl={int((y_te == 0).sum())})")


def get_cols(X_any, genes):
    genes = [str(g) for g in genes]
    missing = [g for g in genes if g not in _col]
    if missing:
        raise KeyError(f"genes absent from the model matrix: {missing}")
    return X_any[:, [_col[g] for g in genes]]


def in_model_space(genes):
    return [str(g) for g in genes if str(g) in _col]


def bootstrap_auc_ci(y_true, y_score, n_boot=N_BOOT, seed=RANDOM_STATE):
    rng, aucs, n = np.random.default_rng(seed), [], len(y_true)
    for _ in range(n_boot):
        i = rng.integers(0, n, size=n)
        if len(np.unique(y_true[i])) < 2:
            continue
        f, t, _ = roc_curve(y_true[i], y_score[i])
        aucs.append(auc(f, t))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def fit_lr_panel(genes, X_tr, X_te_, y_tr):
    sc_ = StandardScaler()
    x_tr = sc_.fit_transform(get_cols(X_tr, genes))
    x_te = sc_.transform(get_cols(X_te_, genes))
    lr = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000,
                            random_state=RANDOM_STATE)
    lr.fit(x_tr, y_tr)
    return lr.predict_proba(x_te)[:, 1]


def panel_roc(genes, X_tr, X_te_, y_tr, y_te_, n_boot=N_BOOT):
    if len(genes) == 0:
        return None
    prob = fit_lr_panel(genes, X_tr, X_te_, y_tr)
    fpr, tpr, _ = roc_curve(y_te_, prob)
    lo, hi = bootstrap_auc_ci(y_te_, prob, n_boot=n_boot)
    return dict(fpr=fpr, tpr=tpr, auc=auc(fpr, tpr), ci_lo=lo, ci_hi=hi, prob=prob)


def single_gene_roc(gene, X_tr, X_te_, y_tr, y_te_, n_boot=N_BOOT):
    sc_ = StandardScaler()
    x_tr = sc_.fit_transform(get_cols(X_tr, [gene]))
    x_te = sc_.transform(get_cols(X_te_, [gene]))
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(x_tr, y_tr)
    prob = lr.predict_proba(x_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te_, prob)
    flipped = auc(fpr, tpr) < 0.5
    if flipped:
        prob = 1 - prob
        fpr, tpr, _ = roc_curve(y_te_, prob)
    lo, hi = bootstrap_auc_ci(y_te_, prob, n_boot=n_boot)
    return dict(fpr=fpr, tpr=tpr, auc=auc(fpr, tpr), ci_lo=lo, ci_hi=hi, flipped=flipped)

print("\nHelpers defined.")

In [ ]:
# ============================== PANEL + SINGLE-GENE ROC ==============================
top5_boruta_genes = importance_df["gene"].astype(str).head(5).tolist()
gene_pool         = [str(g) for g in selected_genes]
N_POOL            = len(gene_pool)
BORUTA_NAME       = f"Boruta {N_POOL}"

ov_roc  = panel_roc(overlap_genes,     X_tr_full, X_te, y_tr_full, y_te)
t5_roc  = panel_roc(top5_boruta_genes, X_tr_full, X_te, y_tr_full, y_te)
b20_roc = panel_roc(gene_pool,         X_tr_full, X_te, y_tr_full, y_te)

def _line(name, r):
    if r is None:
        print(f"{name:24s} (empty - no genes)")
    else:
        print(f"{name:24s} AUC = {r['auc']:.3f}  [{r['ci_lo']:.3f}-{r['ci_hi']:.3f}]")
_line(f"Overlap panel ({len(overlap_genes)})", ov_roc)
_line("Top-5 SHAP panel", t5_roc)
_line(f"Full Boruta panel ({N_POOL})", b20_roc)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for panel_name, genes in [(BORUTA_NAME, gene_pool), ("Overlap", overlap_genes),
                          ("Top-5 SHAP", top5_boruta_genes)]:
    if len(genes) == 0:
        continue
    Xp = get_cols(X_tr_full, genes)
    for clf_name, clf in [
        ("RandomForest", RandomForestClassifier(n_estimators=800, max_depth=10, min_samples_split=4,
                                                min_samples_leaf=2, class_weight="balanced",
                                                random_state=RANDOM_STATE, n_jobs=-1)),
        ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]:
        scores = cross_val_score(clf, StandardScaler().fit_transform(Xp), y_tr_full,
                                 cv=cv, scoring="roc_auc", n_jobs=-1)
        cv_rows.append({"panel": panel_name, "classifier": clf_name,
                        "cv_auc_mean": scores.mean(), "cv_auc_std": scores.std()})
        print(f"  5-fold CV  {panel_name:11s} {clf_name:19s} AUC = {scores.mean():.3f} +/- {scores.std():.3f}")

def _row(name, genes, r):
    return {"panel": name, "n_genes": len(genes),
            "test_auc": r["auc"] if r else np.nan, "ci_lo": r["ci_lo"] if r else np.nan,
            "ci_hi": r["ci_hi"] if r else np.nan, "genes": " | ".join(genes)}
panel_summary = pd.DataFrame([_row("Boruta 20", gene_pool, b20_roc),
                              _row("Overlap 5", overlap_genes, ov_roc),
                              _row("Top-5 SHAP", top5_boruta_genes, t5_roc)])
save_csv(panel_summary, "07_panel_auc_summary.csv",
         "Held-out test AUC with bootstrap 95% CI for each predefined panel "
         "('Boruta 20' = the full Boruta panel, 'Overlap 5' = Boruta and DE set)")
save_csv(pd.DataFrame(cv_rows), "07_panel_crossval_auc.csv",
         "Stratified 5-fold CV AUC per panel on train+val, RF vs logistic regression")

rows, gene_roc = [], {}
for g in gene_pool:
    r = single_gene_roc(g, X_tr_full, X_te, y_tr_full, y_te)
    gene_roc[g] = r
    rows.append({"gene": g, "symbol": sym(g), "auc": r["auc"], "ci_lo": r["ci_lo"],
                 "ci_hi": r["ci_hi"], "direction_flipped": r["flipped"],
                 "in_top5_shap": g in top5_boruta_genes, "in_overlap": g in overlap_genes})
single_df = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)
save_csv(single_df, "07_single_gene_auc.csv",
         f"Per-gene held-out AUC with bootstrap CI for all {N_POOL} Boruta genes")
show_table(single_df, f"Individual gene AUCs (all {N_POOL} Boruta genes):")

In [ ]:
# ---------- Figure: panel ROC comparison ----------
from matplotlib.lines import Line2D
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
def fmt(ax, letter, title):
    ax.set_title(f"{letter}  {title}", fontsize=11, fontweight="bold", loc="left", pad=4)
    ax.set_xlabel("1 - Specificity (FPR)"); ax.set_ylabel("Sensitivity (TPR)")
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.08)
    ax.plot([0, 1], [0, 1], color=C_CHANCE, lw=1, ls="--")
ax = axes[0]; fmt(ax, "A", "Individual gene ROCs")
handles = [Line2D([0], [0], color="none", label="-- Top-5 SHAP (solid) --")]
for i, g in enumerate(top5_boruta_genes):
    r = gene_roc[g]
    ax.plot(r["fpr"], r["tpr"], color=TOP5_COLORS[i], lw=2)
    handles.append(Line2D([0], [0], color=TOP5_COLORS[i], lw=2, label=f"{sym(g)}  {r['auc']:.3f}"))
handles.append(Line2D([0], [0], color="none", label="-- Overlap (dashed) --"))
for i, g in enumerate(overlap_genes[:len(OVERLAP_COLORS)]):
    r = gene_roc[g]
    ax.plot(r["fpr"], r["tpr"], color=OVERLAP_COLORS[i], lw=2, ls="--")
    handles.append(Line2D([0], [0], color=OVERLAP_COLORS[i], lw=2, ls="--", label=f"{sym(g)}  {r['auc']:.3f}"))
ax.legend(handles=handles, loc="lower right", fontsize=7)
ax = axes[1]; fmt(ax, "B", "Panel ROC comparison")
for r, c, lbl in [(t5_roc, C_TOP5, "Top-5 SHAP"), (ov_roc, C_OVERLAP, "Overlap"), (b20_roc, C_DIST, BORUTA_NAME)]:
    if r is not None:
        ax.plot(r["fpr"], r["tpr"], color=c, lw=2.5,
                label=f"{lbl}  AUC = {r['auc']:.3f} [{r['ci_lo']:.3f}-{r['ci_hi']:.3f}]")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
save_fig(fig, "fig05_panel_roc")

## 8. Exhaustive 5-gene panel search

Every five-gene combination of the Boruta panel, each scored by logistic
regression on the held-out test people.

> As in the original, the winner is chosen **on the test set**, so its AUC is an
> optimistic upper bound. §14C reports the nested-CV value for the same procedure.

In [ ]:
# ============================== EXHAUSTIVE SEARCH ==============================
all_combos = list(combinations(gene_pool, PANEL_SIZE))
print(f"Evaluating {len(all_combos):,} combinations ...")
combo_results = []
for n, combo in enumerate(all_combos, 1):
    combo = list(combo)
    prob = fit_lr_panel(combo, X_tr_full, X_te, y_tr_full)
    fpr_, tpr_, _ = roc_curve(y_te, prob)
    combo_results.append({"genes": combo, "auc": auc(fpr_, tpr_), "fpr": fpr_, "tpr": tpr_, "prob": prob})
    if n % 5000 == 0:
        print(f"  {n:,}/{len(all_combos):,}")

# ties are common with 10 test people; break them by the mean single-gene AUC
_sg_auc = {g: gene_roc[g]["auc"] for g in gene_pool}
combo_results.sort(key=lambda d: (d["auc"], np.mean([_sg_auc[g] for g in d["genes"]])), reverse=True)
best       = combo_results[0]
best_genes = [str(g) for g in best["genes"]]
best_auc   = best["auc"]
ci_lo_best, ci_hi_best = bootstrap_auc_ci(y_te, best["prob"])
n_tied_best = int(sum(r["auc"] == best_auc for r in combo_results))

overlap_rank = next((i + 1 for i, r in enumerate(combo_results)
                     if set(map(str, r["genes"])) == set(overlap_genes)), None)
top5_rank    = next((i + 1 for i, r in enumerate(combo_results)
                     if set(map(str, r["genes"])) == set(top5_boruta_genes)), None)

print(f"\nBest panel AUC = {best_auc:.4f}  [{ci_lo_best:.3f}-{ci_hi_best:.3f}]  "
      f"({n_tied_best:,} combinations share this AUC)")
for i, g in enumerate(best_genes, 1):
    flags = [f for f, s in [("overlap", overlap_genes), ("top-5 SHAP", top5_boruta_genes)] if g in s]
    print(f"  {i}. {sym(g)}{'  <- ' + ', '.join(flags) if flags else ''}")
fmt_rank = lambda r: f"{r:,}" if r is not None else "n/a (not a 5-gene subset of the pool)"
print(f"\nOverlap panel rank : {fmt_rank(overlap_rank)} / {len(combo_results):,}")
print(f"Top-5 panel rank   : {fmt_rank(top5_rank)} / {len(combo_results):,}")

top20 = pd.DataFrame([{"rank": i + 1, "auc": r["auc"], "genes": " | ".join(map(str, r["genes"])),
                       "symbols": " | ".join(sym(g) for g in r["genes"])}
                      for i, r in enumerate(combo_results[:20])])
save_csv(top20, "08_exhaustive_top20_panels.csv",
         f"Top 20 of {len(all_combos):,} five-gene panels by held-out AUC")
aucs_all = np.array([r["auc"] for r in combo_results])
save_csv(pd.DataFrame({
    "statistic": ["n_combinations", "mean_auc", "median_auc", "std_auc", "min_auc", "max_auc",
                  "overlap_panel_auc", "overlap_panel_rank", "top5_panel_auc", "top5_panel_rank",
                  "pct_above_overlap", "n_tied_at_max"],
    "value": [len(aucs_all), aucs_all.mean(), np.median(aucs_all), aucs_all.std(), aucs_all.min(),
              aucs_all.max(), ov_roc["auc"] if ov_roc else np.nan,
              overlap_rank if overlap_rank is not None else np.nan, t5_roc["auc"],
              top5_rank if top5_rank is not None else np.nan,
              100 * (aucs_all > ov_roc["auc"]).mean() if ov_roc else np.nan, n_tied_best],
}), "08_exhaustive_auc_distribution.csv", "Summary statistics of the AUC distribution across all 5-gene combinations")
save_csv(pd.DataFrame({"rank": range(1, len(aucs_all) + 1), "auc": aucs_all}),
         "08_exhaustive_all_aucs.csv", "AUC of every 5-gene combination, rank-ordered")

In [ ]:
# ---------- Figure: exhaustive search ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))
ax = axes[0]
ax.hist(aucs_all, bins=40, color=C_DIST, alpha=0.85, edgecolor="white", linewidth=0.3)
for val, c, lbl in [(best_auc, C_BEST, f"optimal {best_auc:.3f}"),
                    (ov_roc["auc"] if ov_roc else None, C_OVERLAP, "overlap"),
                    (t5_roc["auc"], C_TOP5, f"top-5 {t5_roc['auc']:.3f}")]:
    if val is not None:
        ax.axvline(val, color=c, lw=2, ls="--", label=lbl)
ax.set_xlabel("Held-out AUC"); ax.set_ylabel("Number of panels")
ax.set_title(f"A  AUC across all {len(aucs_all):,} panels", fontweight="bold", loc="left")
ax.legend(fontsize=8.5)
ax = axes[1]
ax.plot([0, 1], [0, 1], color=C_CHANCE, lw=1, ls="--")
for r, c, lbl in [(best, C_BEST, "Optimal"), (ov_roc, C_OVERLAP, "Overlap"), (t5_roc, C_TOP5, "Top-5 SHAP")]:
    if r is not None:
        ax.plot(r["fpr"], r["tpr"], color=c, lw=2.5, label=f"{lbl}  {r['auc']:.3f}")
ax.set_title("B  Optimal vs predefined panels", fontweight="bold", loc="left")
ax.legend(loc="lower right", fontsize=9)
ax = axes[2]
freq = pd.Series([g for r in combo_results[:100] for g in map(str, r["genes"])]).value_counts().head(15)
ax.barh(range(len(freq)), freq.values, color=C_DIST, height=0.7)
ax.set_yticks(range(len(freq))); ax.set_yticklabels([sym(g) for g in freq.index], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("Appearances in top 100 panels")
ax.set_title("C  Gene recurrence among best panels", fontweight="bold", loc="left")
fig.tight_layout()
save_fig(fig, "fig08_exhaustive_search")
save_csv(freq.rename_axis("gene").reset_index(name="count_in_top100"),
         "08_gene_recurrence_top100.csv", "How often each gene appears in the 100 best 5-gene panels")

## 9. Optimal panel vs top-5 DE panel

Tests whether ranking genes purely by statistical significance produces a panel
as good as the multivariate search.

In [ ]:
# ============================== OPTIMAL vs TOP-5 DEG ==============================
top5_deg_genes = in_model_space(de_results.sort_values("pvalue")["gene"].astype(str).tolist())[:5]
print("Top-5 DE genes (by p-value):")
for i, g in enumerate(top5_deg_genes, 1):
    r = de_lookup.loc[g]
    print(f"  {i}. {sym(g)}  p={r['pvalue']:.2e}  padj={r['padj']:.2f}  log2FC={r['log2FC']:+.3f}")

deg5_roc = panel_roc(top5_deg_genes, X_tr_full, X_te, y_tr_full, y_te)
print(f"\nTop-5 DE panel AUC = {deg5_roc['auc']:.3f} [{deg5_roc['ci_lo']:.3f}-{deg5_roc['ci_hi']:.3f}]")
print(f"Optimal panel  AUC = {best_auc:.3f} [{ci_lo_best:.3f}-{ci_hi_best:.3f}]")

comp_rows = [
    {"panel": "Optimal (exhaustive)", "auc": best_auc, "ci_lo": ci_lo_best, "ci_hi": ci_hi_best,
     "genes": " | ".join(best_genes)},
    {"panel": "Top-5 SHAP", "auc": t5_roc["auc"], "ci_lo": t5_roc["ci_lo"], "ci_hi": t5_roc["ci_hi"],
     "genes": " | ".join(top5_boruta_genes)},
    {"panel": "Top-5 DEG", "auc": deg5_roc["auc"], "ci_lo": deg5_roc["ci_lo"], "ci_hi": deg5_roc["ci_hi"],
     "genes": " | ".join(top5_deg_genes)},
]
if ov_roc is not None:
    comp_rows.insert(1, {"panel": "Overlap (Boruta n DEG)", "auc": ov_roc["auc"], "ci_lo": ov_roc["ci_lo"],
                         "ci_hi": ov_roc["ci_hi"], "genes": " | ".join(overlap_genes)})
comparison = pd.DataFrame(comp_rows).sort_values("auc", ascending=False).reset_index(drop=True)
save_csv(comparison, "09_panel_comparison.csv", "Head-to-head AUC of the candidate panels with bootstrap CIs")

panels = {"Optimal": best_genes, "Overlap": overlap_genes,
          "Top-5 SHAP": top5_boruta_genes, "Top-5 DEG": top5_deg_genes}
union = sorted({g for gs in panels.values() for g in gs})
mat = pd.DataFrame({"gene": union})
for name, gs in panels.items():
    mat[name] = mat["gene"].isin(gs)
mat["n_panels"] = mat[list(panels)].sum(axis=1)
mat["symbol"] = mat["gene"].map(sym)
mat = mat.sort_values("n_panels", ascending=False).reset_index(drop=True)
save_csv(mat, "09_panel_membership_matrix.csv", "Which genes appear in which candidate panel")
print(f"\nGenes in >= 3 panels: {[sym(g) for g in mat.loc[mat['n_panels'] >= 3, 'gene']]}")

In [ ]:
# ---------- Figure: optimal vs DEG-5 ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
ax = axes[0]
ax.plot([0, 1], [0, 1], color=C_CHANCE, lw=1, ls="--")
for r, c, lbl in [(best, C_BEST, "Optimal"), (deg5_roc, C_DEG5, "Top-5 DE"), (ov_roc, C_OVERLAP, "Overlap")]:
    if r is not None:
        ax.plot(r["fpr"], r["tpr"], color=c, lw=2.5, label=f"{lbl}  AUC = {r['auc']:.3f}")
ax.set_title("A  ML-optimised vs significance-ranked panels", fontweight="bold", loc="left")
ax.legend(loc="lower right", fontsize=9)
ax = axes[1]
hm = mat.set_index("symbol")[list(panels)].astype(int)
sns.heatmap(hm, cmap=["#F5F5F5", C_BEST], cbar=False, linewidths=1.2, linecolor="white", ax=ax)
ax.set_title("B  Panel membership", fontweight="bold", loc="left"); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
save_fig(fig, "fig10_optimal_vs_deg")

## 10. External validation on GSE7621

Independent bulk substantia nigra cohort (GPL570, 9 control / 16 PD). The panel's
Ensembl IDs are mapped to Affymetrix probes with g:Profiler; where a gene has
several probes the one with the highest mean expression is used.

Two questions, kept apart:

1. **Transfer** — the logistic model trained on all 63 merged people is applied
   **unchanged** to GSE7621 (each gene z-scored within GSE7621, the same
   label-free step as §1). Nothing is refitted on the external cohort.
2. **Within-cohort** — as in the original: a Random Forest fitted on 60% of
   GSE7621 using the panel genes, scored on the other 40%, with repeated CV and
   importance averaged over 40 seeds.

GSE7621 is bulk tissue, and PD nigra has lost most of its dopamine neurons. A
panel can therefore separate the groups by tracking **neuron content** rather
than a change inside neurons. The transfer test is repeated after removing the
part of the score explained by eight dopamine-neuron marker genes.

In [ ]:
# ============================== GSE7621 EXTERNAL VALIDATION ==============================
gse_path = find_input(F_GSE7621)
print("GSE7621 file:", gse_path)
with open(gse_path, encoding="utf-8", errors="replace") as fh:
    gse_lines = fh.readlines()
title_line = next(l for l in gse_lines if l.startswith("!Sample_title"))
titles = [t.strip('"') for t in title_line.rstrip("\n").split("\t")[1:]]
y_ext = np.array([0 if "normal" in t.lower() else 1 for t in titles])
print(f"Labels from header: Control={int((y_ext == 0).sum())}, PD={int(y_ext.sum())}")

start = next(i for i, l in enumerate(gse_lines) if "!series_matrix_table_begin" in l)
rows = []
for line in gse_lines[start + 1:]:
    if "!series_matrix_table_end" in line:
        break
    r = [x.strip('"') for x in line.rstrip("\n").split("\t")]
    if r and r[0] and not r[0].startswith("!"):
        rows.append(r)
expr_df = (pd.DataFrame(rows[1:], columns=rows[0]).set_index(rows[0][0])
             .apply(pd.to_numeric, errors="coerce").T)
print(f"Loaded GSE7621: {expr_df.shape[0]} samples x {expr_df.shape[1]} probes")
log_ext_all = np.log2(expr_df.clip(lower=1.0))
probe_mean = log_ext_all.mean(axis=0)

def ens_to_probes(ens_ids):
    """Ensembl -> GPL570 probes present in the file, best (highest mean) first."""
    conv, _ = gconvert(list(ens_ids), "AFFY_HG_U133_PLUS_2")
    out = {}
    for g, ps in conv.items():
        ps = sorted([p for p in ps if p in probe_mean.index], key=lambda p: -probe_mean[p])
        if ps:
            out[g] = ps
    return out

PANEL_PROBES = ens_to_probes(gene_pool)
GENE_PROBES  = {sym(g): PANEL_PROBES[g] for g in gene_pool if g in PANEL_PROBES}
selected_probes, probe_to_gene = [], {}
for g in gene_pool:
    if g in PANEL_PROBES:
        selected_probes.append(PANEL_PROBES[g][0]); probe_to_gene[PANEL_PROBES[g][0]] = sym(g)
    else:
        print(f"  WARNING: no GPL570 probe for {sym(g)}")
X_ext = expr_df[selected_probes].values
print(f"Using {len(selected_probes)}/{N_POOL} panel genes | {len(y_ext)} samples")

# gene-level, label-free z-scores for the transfer test
def ext_gene_z(ens_ids):
    pm = ens_to_probes(ens_ids)
    cols = {g: log_ext_all[ps[0]].to_numpy(float) for g, ps in pm.items()}
    E = pd.DataFrame(cols)
    return (E - E.mean(0)) / E.std(0, ddof=1).clip(lower=0.05)
EZ = ext_gene_z(set(gene_pool) | set(top5_deg_genes))

In [ ]:
# ---------- 1. transfer: trained on the 63 merged people, applied unchanged ----------
def perm_p(y_true, s, n=5000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); a = roc_auc_score(y_true, s)
    null = np.array([roc_auc_score(rng.permutation(y_true), s) for _ in range(n)])
    return float((np.sum(null >= a) + 1) / (n + 1))

TRANSFER_PANELS = {"Boruta 20": gene_pool, "Top-5 SHAP": top5_boruta_genes,
                   "Top-5 DEG": top5_deg_genes, "Optimal (exhaustive)": best_genes}
if overlap_genes:
    TRANSFER_PANELS["Overlap (Boruta n DEG)"] = overlap_genes
transfer_rows, transfer_scores = [], {}
for name, genes in TRANSFER_PANELS.items():
    gm = [g for g in genes if g in EZ.columns]
    if len(gm) < 1:
        continue
    m = make_lr = LogisticRegression(C=1.0, max_iter=3000)
    sc_ = StandardScaler().fit(get_cols(X, gm))
    make_lr.fit(sc_.transform(get_cols(X, gm)), y)
    s = make_lr.predict_proba(sc_.transform(EZ[gm].to_numpy()))[:, 1]
    transfer_scores[name] = s
    lo, hi = bootstrap_auc_ci(y_ext, s)
    transfer_rows.append({"panel": name, "genes_mapped": len(gm), "genes_in_panel": len(genes),
                          "auc": roc_auc_score(y_ext, s), "ci_lo": lo, "ci_hi": hi,
                          "perm_p": perm_p(y_ext, s)})
transfer_df = pd.DataFrame(transfer_rows)

# ---- is the transfer explained by neuron loss? ------------------------------
MARKERS = ["TH", "SLC6A3", "DDC", "SLC18A2", "KCNJ6", "ALDH1A1", "NR4A2", "EN1"]
m_conv, _ = gconvert(MARKERS, "ENSG")
m_ens = [sorted(m_conv[s])[0] for s in MARKERS if s in m_conv]
MZ = ext_gene_z(m_ens)
neuron_score = MZ.mean(axis=1).to_numpy()
neuron_auc = roc_auc_score(1 - y_ext, neuron_score)
adj_rows = []
for name, s in transfer_scores.items():
    resid = s - np.polyval(np.polyfit(neuron_score, s, 1), neuron_score)
    adj_rows.append({"panel": name, "auc": roc_auc_score(y_ext, s),
                     "auc_after_neuron_adjustment": roc_auc_score(y_ext, resid),
                     "perm_p_adjusted": perm_p(y_ext, resid),
                     "spearman_with_neuron_score": float(stats.spearmanr(s, neuron_score)[0])})
adj_df = pd.DataFrame(adj_rows)
print(f"Dopamine-neuron marker score ({MZ.shape[1]} genes) separates control > PD with AUC {neuron_auc:.3f}")
save_csv(transfer_df, "10_external_transfer_gse7621.csv",
         "Model trained on the 63 merged people applied unchanged to GSE7621")
save_csv(adj_df.assign(neuron_marker_auc=neuron_auc, n_markers=MZ.shape[1]),
         "10_external_transfer_neuron_adjusted.csv",
         "Transfer AUC before and after removing the dopamine-neuron marker trend")
show_table(transfer_df.merge(adj_df[["panel", "auc_after_neuron_adjustment", "perm_p_adjusted"]], on="panel"),
           "Transfer to GSE7621 (no refitting):")

In [ ]:
# ---------- 2. within-cohort, as in the original ----------
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(X_ext, y_ext, test_size=0.4, stratify=y_ext,
                                              random_state=RANDOM_STATE)
print(f"Train {len(ye_tr)} | Test {len(ye_te)}")
rf_ext = RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_split=5,
                                random_state=RANDOM_STATE, n_jobs=-1)
rf_ext.fit(Xe_tr, ye_tr)
prob_ext = rf_ext.predict_proba(Xe_te)[:, 1]
auc_ext  = roc_auc_score(ye_te, prob_ext)
ci_ext   = bootstrap_auc_ci(ye_te, prob_ext)
cv_ext = cross_val_score(rf_ext, X_ext, y_ext,
                         cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=30, random_state=RANDOM_STATE),
                         scoring="roc_auc", n_jobs=-1)
print(f"\nHold-out test AUC = {auc_ext:.4f}  [{ci_ext[0]:.3f}-{ci_ext[1]:.3f}]  (n={len(ye_te)})")
print(f"Repeated CV AUC   = {cv_ext.mean():.4f} +/- {cv_ext.std():.4f}  (5-fold x 30)")

bt = transfer_df.set_index("panel").loc["Boruta 20"]
ba = adj_df.set_index("panel").loc["Boruta 20"]
save_csv(pd.DataFrame([
    {"metric": "holdout_auc",       "value": auc_ext,        "n": len(ye_te)},
    {"metric": "holdout_ci_lo",     "value": ci_ext[0],      "n": len(ye_te)},
    {"metric": "holdout_ci_hi",     "value": ci_ext[1],      "n": len(ye_te)},
    {"metric": "repeated_cv_mean",  "value": cv_ext.mean(),  "n": len(y_ext)},
    {"metric": "repeated_cv_std",   "value": cv_ext.std(),   "n": len(y_ext)},
    {"metric": "transfer_auc",      "value": bt["auc"],      "n": len(y_ext)},
    {"metric": "transfer_ci_lo",    "value": bt["ci_lo"],    "n": len(y_ext)},
    {"metric": "transfer_ci_hi",    "value": bt["ci_hi"],    "n": len(y_ext)},
    {"metric": "transfer_perm_p",   "value": bt["perm_p"],   "n": len(y_ext)},
    {"metric": "transfer_auc_neuron_adjusted", "value": ba["auc_after_neuron_adjustment"], "n": len(y_ext)},
    {"metric": "neuron_marker_auc", "value": neuron_auc,     "n": len(y_ext)},
    {"metric": "n_probes_mapped",   "value": len(selected_probes), "n": N_POOL},
    {"metric": "n_control",         "value": int((y_ext == 0).sum()), "n": len(y_ext)},
    {"metric": "n_pd",              "value": int(y_ext.sum()),       "n": len(y_ext)},
]), "10_external_validation_gse7621.csv",
   "GSE7621: transfer AUC (no refitting), within-cohort hold-out AUC, CI and repeated CV")

N_IMP_SEEDS = 40
imp_matrix, auc_by_seed = [], []
for seed in range(N_IMP_SEEDS):
    rf_s = RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_split=5,
                                  random_state=seed, n_jobs=-1).fit(Xe_tr, ye_tr)
    imp_matrix.append(rf_s.feature_importances_)
    auc_by_seed.append(roc_auc_score(ye_te, rf_s.predict_proba(Xe_te)[:, 1]))
imp_matrix, auc_by_seed = np.asarray(imp_matrix), np.asarray(auc_by_seed)
rank_matrix = np.asarray([pd.Series(r).rank(ascending=False).values for r in imp_matrix])
ext_imp = pd.DataFrame({
    "probe": selected_probes, "gene": [probe_to_gene[p] for p in selected_probes],
    "importance_seed42": rf_ext.feature_importances_, "importance": imp_matrix.mean(axis=0),
    "importance_std": imp_matrix.std(axis=0), "rank_mean": rank_matrix.mean(axis=0),
    "rank_min": rank_matrix.min(axis=0).astype(int), "rank_max": rank_matrix.max(axis=0).astype(int),
}).sort_values("importance", ascending=False).reset_index(drop=True)
ext_imp["rank"] = range(1, len(ext_imp) + 1)
ext_imp["rank_span"] = ext_imp["rank_max"] - ext_imp["rank_min"]
save_csv(ext_imp, "10_gse7621_feature_importance.csv",
         f"GSE7621 RF importance per gene, averaged over {N_IMP_SEEDS} seeds")
save_csv(ext_imp[["gene", "rank", "rank_mean", "rank_min", "rank_max", "rank_span",
                  "importance", "importance_std", "importance_seed42"]],
         "10_gse7621_importance_stability.csv", "Rank stability of each gene across seeds")
save_csv(pd.DataFrame({"seed": range(N_IMP_SEEDS), "holdout_auc": auc_by_seed}),
         "10_gse7621_auc_by_seed.csv", "Hold-out AUC for each random seed")
print(f"\nAUC across {N_IMP_SEEDS} seeds: mean={auc_by_seed.mean():.4f}")
show_table(ext_imp[["gene", "rank", "importance", "importance_std", "rank_min", "rank_max"]],
           f"GSE7621 gene importance (averaged over {N_IMP_SEEDS} seeds):")

In [ ]:
# ---------- Figure: external validation ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
ax = axes[0]
fpr_t, tpr_t, _ = roc_curve(y_ext, transfer_scores["Boruta 20"])
ax.plot(fpr_t, tpr_t, color=C_BEST, lw=2.5, label=f"Transfer (no refit)  AUC = {bt['auc']:.3f}")
fpr_e, tpr_e, _ = roc_curve(ye_te, prob_ext)
ax.plot(fpr_e, tpr_e, color=C_DEG5, lw=2, ls="--", label=f"Within-cohort hold-out  AUC = {auc_ext:.3f}")
ax.plot([0, 1], [0, 1], color=C_CHANCE, lw=1, ls="--")
ax.set_xlabel("1 - Specificity"); ax.set_ylabel("Sensitivity")
ax.set_title(f"A  GSE7621, {N_POOL}-gene panel", fontweight="bold", loc="left")
ax.legend(loc="lower right", fontsize=9)
ax = axes[1]
top15 = ext_imp.head(15)
ax.barh(np.arange(len(top15)), top15["importance"], color=C_DEG5, height=0.72,
        xerr=top15["importance_std"], error_kw=dict(ecolor="#555555", elinewidth=0.8, capsize=2.5))
ax.set_yticks(np.arange(len(top15))); ax.set_yticklabels(top15["gene"], fontsize=9)
ax.invert_yaxis(); ax.set_xlabel("Feature importance (MDI)")
ax.set_title(f"B  Top genes inside GSE7621 (mean +/- SD, {N_IMP_SEEDS} seeds)", fontweight="bold", loc="left")
fig.tight_layout()
save_fig(fig, "fig11_external_validation")

## 10b. The same selection, run again on the external cohort

Boruta is run from scratch on GSE7621 with the same settings as §4, over its
8,000 most variable probes. With 25 samples it is the weaker selection by
construction, so the intersection is a floor, not a ceiling.

In [ ]:
# ============================== EXTERNAL BORUTA ==============================
N_EXT_HVG = 8000
probe_var  = log_ext_all.var(axis=0).sort_values(ascending=False)
hvg_probes = list(probe_var.head(min(N_EXT_HVG, len(probe_var))).index)
X_ext_all  = log_ext_all[hvg_probes].to_numpy(float)
print(f"External selection matrix: {X_ext_all.shape[0]} samples x {X_ext_all.shape[1]} probes")
ext_selector = BorutaPy(
    estimator=RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_split=4,
                                     min_samples_leaf=2, class_weight="balanced", n_jobs=-1,
                                     random_state=RANDOM_STATE),
    n_estimators="auto", perc=BORUTA_PERC, alpha=BORUTA_ALPHA, two_step=True,
    max_iter=BORUTA_MAX_ITER, random_state=RANDOM_STATE, verbose=0)
ext_selector.fit(X_ext_all, y_ext)
ext_conf_probes = [p for p, k in zip(hvg_probes, ext_selector.support_) if k]
ext_tent_probes = [p for p, k in zip(hvg_probes, ext_selector.support_weak_) if k]
print(f"GSE7621 Boruta: {len(ext_conf_probes)} confirmed, {len(ext_tent_probes)} tentative")

In [ ]:
# ---------- probe -> symbol, so the two selections are comparable ----------
def probes_to_symbols(probes):
    probes = list(probes)
    if not probes:
        return {}
    conv, names = gconvert(probes, "ENSG")
    out = {}
    for p, gs in conv.items():
        nm = sorted({names.get(g, "") for g in gs} - {""})
        if nm:
            out[p] = nm[0]
    for s_, ps in GENE_PROBES.items():
        for p in ps:
            if p in probes:
                out.setdefault(p, s_)
    return out

p2s = probes_to_symbols(set(ext_conf_probes) | set(ext_tent_probes))
ext_conf_syms = sorted({p2s[p] for p in ext_conf_probes if p in p2s})
ext_tent_syms = sorted({p2s[p] for p in ext_tent_probes if p in p2s})
disc_syms     = sorted(sym(g) for g in gene_pool)
inter_conf = sorted(set(ext_conf_syms) & set(disc_syms))
inter_any  = sorted((set(ext_conf_syms) | set(ext_tent_syms)) & set(disc_syms))
from scipy.stats import hypergeom
BG = len(hvg_probes)
hyper_p = float(hypergeom.sf(len(inter_conf) - 1, BG, len(ext_conf_syms), len(disc_syms))) \
          if ext_conf_syms else float("nan")
ext_boruta_df = pd.DataFrame({
    "probe": ext_conf_probes + ext_tent_probes,
    "symbol": [p2s.get(p, "") for p in ext_conf_probes + ext_tent_probes],
    "status": ["confirmed"] * len(ext_conf_probes) + ["tentative"] * len(ext_tent_probes)})
ext_boruta_df["in_discovery_panel"] = ext_boruta_df["symbol"].isin(disc_syms)
save_csv(ext_boruta_df, "10b_gse7621_boruta_genes.csv", "Boruta re-run from scratch on GSE7621, mapped to symbols")
union_sz = len(set(ext_conf_syms) | set(disc_syms))
save_csv(pd.DataFrame([
    ("n_background_probes", BG), ("n_external_samples", int(len(y_ext))),
    ("n_discovery_selected", len(disc_syms)), ("n_external_confirmed", len(ext_conf_syms)),
    ("n_external_tentative", len(ext_tent_syms)), ("n_intersection_confirmed", len(inter_conf)),
    ("n_intersection_incl_tentative", len(inter_any)),
    ("jaccard_confirmed", (len(inter_conf) / union_sz) if union_sz else 0.0), ("hypergeometric_p", hyper_p),
], columns=["statistic", "value"]), "10b_boruta_cohort_overlap.csv",
   "Overlap between the discovery and external Boruta selections")
print(f"Intersection: {len(inter_conf)} confirmed" + (f" -> {', '.join(inter_conf)}" if inter_conf else ""))
print(f"Hypergeometric p = {hyper_p:.3g}")

## 10c. Where the panel ranks in the external cohort

A Random Forest on **every** GSE7621 probe, averaged over ten seeds: the external
top 20, and where each panel gene sits among all probes. §10d repeats it with
SHAP.

In [ ]:
# ============================== EXTERNAL GLOBAL RANKING ==============================
N_RANK_SEEDS = 10
all_probes = list(expr_df.columns)
X_full = np.log2(expr_df.clip(lower=1.0) + 1.0).to_numpy(float)
imp_acc = np.zeros(X_full.shape[1])
for _s in range(N_RANK_SEEDS):
    rf_r = RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_split=4, min_samples_leaf=2,
                                  class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE + _s)
    imp_acc += rf_r.fit(X_full, y_ext).feature_importances_
rank_df = pd.DataFrame({"probe": all_probes, "importance": imp_acc / N_RANK_SEEDS})
rank_df = rank_df.sort_values("importance", ascending=False).reset_index(drop=True)
rank_df["rank"] = np.arange(1, len(rank_df) + 1)
N_PROBES = len(rank_df)

top20 = rank_df.head(20).copy()
t20_map = probes_to_symbols(list(top20["probe"]))
top20["symbol"] = [t20_map.get(p, "") for p in top20["probe"]]
top20["in_discovery_panel"] = top20["symbol"].isin(disc_syms)
n_top20_hit = int(top20["in_discovery_panel"].sum())
save_csv(top20[["rank", "probe", "symbol", "importance", "in_discovery_panel"]],
         "10c_gse7621_top20_global.csv", "Top 20 probes by RF importance across the whole GSE7621 array")

rank_by_probe = dict(zip(rank_df["probe"], rank_df["rank"]))
rows_rank = []
for _g, _probes in GENE_PROBES.items():
    hits = [(rank_by_probe[p], p) for p in _probes if p in rank_by_probe]
    if not hits:
        rows_rank.append({"symbol": _g, "probe": "", "rank": np.nan, "percentile": np.nan}); continue
    _r, _pb = min(hits)
    rows_rank.append({"symbol": _g, "probe": _pb, "rank": int(_r),
                      "percentile": 100.0 * (1.0 - (_r - 1) / N_PROBES)})
panel_rank = pd.DataFrame(rows_rank).sort_values("rank").reset_index(drop=True)
save_csv(panel_rank, "10c_panel_global_rank_in_gse7621.csv", "Rank of every discovery panel gene among all GSE7621 probes")
median_pct = float(panel_rank["percentile"].median())
from scipy.stats import wilcoxon
try:
    w_p = float(wilcoxon(panel_rank["percentile"].dropna() - 50.0).pvalue)
except Exception:
    w_p = float("nan")
save_csv(pd.DataFrame([
    ("n_probes_ranked", N_PROBES), ("n_rank_seeds", N_RANK_SEEDS),
    ("n_top20_in_discovery_panel", n_top20_hit), ("panel_median_percentile", median_pct),
    ("panel_genes_top10pct", int((panel_rank["percentile"] >= 90).sum())),
    ("panel_genes_top1pct", int((panel_rank["percentile"] >= 99).sum())), ("wilcoxon_p_vs_uniform", w_p),
], columns=["statistic", "value"]), "10c_external_rank_summary.csv",
   "Where the discovery panel sits in the external importance ranking")
print(f"External top 20 shares {n_top20_hit} gene(s) with the panel; median percentile {median_pct:.1f}")
show_table(panel_rank, "Discovery panel genes ranked inside GSE7621:")

In [ ]:
# ---------- the same ranking, but on SHAP rather than impurity ----------
N_SHAP_SEEDS = 3
shap_acc = np.zeros(X_full.shape[1])
for _s in range(N_SHAP_SEEDS):
    rf_s = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_split=4, min_samples_leaf=2,
                                  class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE + _s)
    rf_s.fit(X_full, y_ext)
    sv_ext = shap.TreeExplainer(rf_s).shap_values(X_full)
    if isinstance(sv_ext, list):
        sv_ext = sv_ext[1]
    elif getattr(sv_ext, "ndim", 2) == 3:
        sv_ext = sv_ext[:, :, 1]
    shap_acc += np.abs(sv_ext).mean(axis=0)
shap_rank = pd.DataFrame({"probe": all_probes, "mean_abs_shap": shap_acc / N_SHAP_SEEDS})
shap_rank = shap_rank.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_rank["rank"] = np.arange(1, len(shap_rank) + 1)
top20s = shap_rank.head(20).copy()
s20_map = probes_to_symbols(list(top20s["probe"]))
top20s["symbol"] = [s20_map.get(p, "") for p in top20s["probe"]]
top20s["in_discovery_panel"] = top20s["symbol"].isin(disc_syms)
n_shap_hit = int(top20s["in_discovery_panel"].sum())
save_csv(top20s[["rank", "probe", "symbol", "mean_abs_shap", "in_discovery_panel"]],
         "10d_gse7621_top20_shap.csv", "Top 20 probes by mean |SHAP| across the whole GSE7621 array")
srank_by_probe = dict(zip(shap_rank["probe"], shap_rank["rank"]))
rows_s = []
for _g, _probes in GENE_PROBES.items():
    hits = [(srank_by_probe[p], p) for p in _probes if p in srank_by_probe]
    if not hits:
        rows_s.append({"symbol": _g, "probe": "", "shap_rank_ext": np.nan, "percentile": np.nan}); continue
    _r, _pb = min(hits)
    rows_s.append({"symbol": _g, "probe": _pb, "shap_rank_ext": int(_r),
                   "percentile": 100.0 * (1.0 - (_r - 1) / len(shap_rank))})
panel_shap_rank = pd.DataFrame(rows_s).sort_values("shap_rank_ext").reset_index(drop=True)
save_csv(panel_shap_rank, "10d_panel_shap_rank_in_gse7621.csv", "SHAP rank of every panel gene among all GSE7621 probes")
med_s = float(panel_shap_rank["percentile"].median())
save_csv(pd.DataFrame([
    ("n_probes_ranked", len(shap_rank)), ("n_shap_seeds", N_SHAP_SEEDS),
    ("n_boruta_in_external_top20", n_shap_hit), ("n_discovery_boruta", len(disc_syms)),
    ("panel_median_shap_percentile", med_s),
    ("panel_genes_top10pct", int((panel_shap_rank["percentile"] >= 90).sum())),
    ("expected_overlap_by_chance", 20 * len(disc_syms) / len(shap_rank)),
], columns=["statistic", "value"]), "10d_shap_top20_overlap.csv",
   "How many Boruta-selected genes reach the external cohort's top 20 by SHAP")
print(f"{n_shap_hit} of {len(disc_syms)} panel genes in GSE7621's SHAP top 20; median percentile {med_s:.1f}")

## 11. Cross-cohort feature stability

SHAP importance in the merged discovery cohort against Random Forest importance
inside GSE7621, both multi-seed averages.

In [ ]:
# ============================== CROSS-COHORT STABILITY ==============================
from scipy.stats import spearmanr, pearsonr
ens_to_symbol = {g: sym(g) for g in gene_pool}
cross = importance_df[["gene", "shap_rank", "mean_abs_shap", "mean_shap", "direction"]].copy()
cross["symbol"] = cross["gene"].map(ens_to_symbol)
ext_lookup = ext_imp.set_index("gene")
for c_out, c_in in (("gse7621_importance", "importance"), ("gse7621_rank", "rank"),
                    ("gse7621_rank_min", "rank_min"), ("gse7621_rank_max", "rank_max")):
    cross[c_out] = cross["symbol"].map(ext_lookup[c_in])
cross = cross[["gene", "symbol", "shap_rank", "mean_abs_shap", "mean_shap", "direction",
               "gse7621_rank", "gse7621_rank_min", "gse7621_rank_max", "gse7621_importance"]]
save_csv(cross, "11_cross_cohort_importance.csv", "Discovery SHAP importance vs GSE7621 RF importance, per gene")
paired = cross.dropna(subset=["gse7621_importance"])
print(f"Genes measurable in both cohorts: {len(paired)}/{len(cross)}")
if len(paired) >= 3:
    rho, p_rho = spearmanr(paired["shap_rank"], paired["gse7621_rank"])
    r, p_r = pearsonr(paired["mean_abs_shap"], paired["gse7621_importance"])
    print(f"Spearman rho = {rho:.3f} (p = {p_rho:.3f}) | Pearson r = {r:.3f} (p = {p_r:.3f})")
else:
    rho = p_rho = r = p_r = float("nan")
save_csv(pd.DataFrame([
    {"statistic": "n_genes_paired", "value": len(paired)}, {"statistic": "spearman_rho", "value": rho},
    {"statistic": "spearman_p", "value": p_rho}, {"statistic": "pearson_r", "value": r},
    {"statistic": "pearson_p", "value": p_r},
]), "11_cross_cohort_correlation.csv", "Rank and linear correlation of gene importance between the two cohorts")

In [ ]:
# ---------- Figure: cross-cohort comparison ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
ax = axes[0]
plot = paired.sort_values("gse7621_importance", ascending=True)
ypos = np.arange(len(plot))
ax.barh(ypos - 0.2, plot["mean_abs_shap"] / plot["mean_abs_shap"].max(), height=0.38, color=C_DEG5,
        label="merged LCM (mean |SHAP|, scaled)")
ax.barh(ypos + 0.2, plot["gse7621_importance"] / plot["gse7621_importance"].max(), height=0.38,
        color="#D6604D", label="GSE7621 (RF importance, scaled)")
ax.set_yticks(ypos); ax.set_yticklabels(plot["symbol"], fontsize=9)
ax.set_title("A  Importance in both cohorts", fontweight="bold", loc="left")
ax.legend(fontsize=8.5)
ax = axes[1]
ax.scatter(paired["shap_rank"], paired["gse7621_rank"], s=70, color=C_BEST, edgecolors="black", linewidths=0.5)
for _, row in paired.iterrows():
    ax.annotate(row["symbol"], (row["shap_rank"], row["gse7621_rank"]), fontsize=7.5,
                xytext=(4, 3), textcoords="offset points")
lims = [0, max(paired["shap_rank"].max(), paired["gse7621_rank"].max()) + 1]
ax.plot(lims, lims, ls="--", lw=1, color=C_CHANCE)
ax.set_xlabel("Rank in merged LCM (SHAP)"); ax.set_ylabel("Rank in GSE7621 (RF importance)")
ax.set_title(f"B  Rank concordance (rho = {rho:.2f})", fontweight="bold", loc="left")
fig.tight_layout()
save_fig(fig, "fig13_cross_cohort")

## 12. Functional enrichment (g:Profiler)

The original notebook read a file exported from the g:Profiler website. Here the
same query is sent to the g:Profiler API: the Boruta panel against a **custom
background of every harmonised gene**, Benjamini-Hochberg corrected, all sources.

In [ ]:
# ============================== ENRICHMENT ==============================
gost_body = {"organism": "hsapiens", "query": gene_pool, "sources": [],
             "user_threshold": 0.05, "all_results": False, "ordered": False,
             "no_evidences": False, "no_iea": False, "combined": False,
             "measure_underrepresentation": False, "domain_scope": "custom",
             "background": GENES, "significance_threshold_method": "fdr"}
gres = None
for k in range(6):
    try:
        req = urllib.request.Request("https://biit.cs.ut.ee/gprofiler/api/gost/profile/",
                                     data=json.dumps(gost_body).encode(),
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=600) as fh:
            gres = json.loads(fh.read())
        break
    except Exception as exc:
        print(f"  g:GOSt retry {k + 1}: {type(exc).__name__}")
        time.sleep(10 * (k + 1))
if gres is None:
    raise RuntimeError("g:Profiler g:GOSt unreachable; is Internet on?")

qgenes = gres["meta"]["genes_metadata"]["query"]["query_1"]["ensgs"]
qgenes = [g[0] if isinstance(g, list) and g else g for g in qgenes]
SOURCE_LABEL = {"GO:CC": "GO: Cellular Component", "GO:BP": "GO: Biological Process",
                "GO:MF": "GO: Molecular Function", "CORUM": "CORUM Complex",
                "WP": "WikiPathways", "KEGG": "KEGG", "REAC": "Reactome",
                "HPA": "Human Protein Atlas", "TF": "Transcription Factor",
                "MIRNA": "miRTarBase", "HP": "Human Phenotype Ontology"}
gp_rows = []
for r in gres["result"]:
    inter = [qgenes[i] for i, ev in enumerate(r.get("intersections", [])) if ev and i < len(qgenes)]
    gp_rows.append({"source": r["source"], "term_name": r["name"], "term_id": r["native"],
                    "adjusted_p_value": r["p_value"], "term_size": r["term_size"],
                    "query_size": r["query_size"], "intersection_size": r["intersection_size"],
                    "effective_domain_size": r["effective_domain_size"],
                    "intersections": ",".join(inter)})
GP_COLS = ["source", "term_name", "term_id", "adjusted_p_value", "term_size", "query_size",
           "intersection_size", "effective_domain_size", "intersections"]
gp = pd.DataFrame(gp_rows, columns=GP_COLS)
gp["source_label"] = gp["source"].map(SOURCE_LABEL).fillna(gp["source"])
gp["negative_log10_of_adjusted_p_value"] = -np.log10(gp["adjusted_p_value"].astype(float))
gp["gene_ratio"] = gp["intersection_size"] / gp["term_size"].clip(lower=1)
gp_sorted = gp.sort_values("negative_log10_of_adjusted_p_value", ascending=False).reset_index(drop=True)
print(f"g:Profiler: {len(gp_sorted)} terms at adjusted p < 0.05 "
      f"(query {len(gene_pool)} genes, background {len(GENES):,})")
cols = ["source_label", "term_name", "term_id", "adjusted_p_value", "negative_log10_of_adjusted_p_value",
        "intersection_size", "term_size", "gene_ratio", "intersections"]
save_csv(gp_sorted[cols], "12_enrichment_summary.csv",
         "g:Profiler enriched terms (custom background, BH), with intersecting genes")
show_table(gp_sorted[["source_label", "term_name", "adjusted_p_value", "intersection_size", "term_size"]],
           "Enriched terms:")

edges = []
for _, row in gp_sorted.iterrows():
    for g in str(row["intersections"]).split(","):
        g = g.strip()
        if g and g.lower() != "nan":
            edges.append({"gene": g, "term": row["term_name"], "source": row["source_label"],
                          "adjusted_p_value": row["adjusted_p_value"]})
edges_df = pd.DataFrame(edges, columns=["gene", "term", "source", "adjusted_p_value"])
save_csv(edges_df, "12_gene_term_edges.csv", "Bipartite gene-to-term edge list for the enrichment network")
hubs = (edges_df.groupby("gene").size().sort_values(ascending=False)
        .rename_axis("gene").reset_index(name="n_terms"))
hubs["symbol"] = hubs["gene"].map(sym)
save_csv(hubs, "12_gene_hub_degree.csv", "Number of enriched terms each gene participates in")

## 13. Gene annotation table

Every per-gene number in one table.

In [ ]:
# ============================== MASTER GENE TABLE ==============================
ann = importance_df[["gene", "shap_rank", "mean_abs_shap", "mean_shap", "shap_std", "shap_cv", "direction"]].copy()
ann["symbol"] = ann["gene"].map(ens_to_symbol)
de_idx = de_results.set_index(de_results["gene"].astype(str))
for c in ("log2FC", "pvalue", "padj", "hedges_g_meta", "I2", "same_direction_datasets"):
    ann[c] = ann["gene"].map(de_idx[c])
ann["is_deg"] = ann["gene"].isin(deg_set)
ann["in_overlap_panel"] = ann["gene"].isin(overlap_genes)
ann["in_top5_shap"]     = ann["gene"].isin(top5_boruta_genes)
ann["in_optimal_panel"] = ann["gene"].isin(best_genes)
ann["in_top5_deg"]      = ann["gene"].isin(top5_deg_genes)
sg = single_df.set_index("gene")
ann["single_gene_auc"]   = ann["gene"].map(sg["auc"])
ann["single_gene_ci_lo"] = ann["gene"].map(sg["ci_lo"])
ann["single_gene_ci_hi"] = ann["gene"].map(sg["ci_hi"])
ann["gse7621_rank"]       = ann["symbol"].map(ext_lookup["rank"])
ann["gse7621_importance"] = ann["symbol"].map(ext_lookup["importance"])
ann["gse7621_probe"]      = ann["gene"].map(lambda g: PANEL_PROBES.get(g, [""])[0])
hub_map = edges_df.groupby("gene").size()
ann["n_enriched_terms"] = ann["gene"].map(hub_map).fillna(0).astype(int)
ann = ann[["shap_rank", "gene", "symbol", "mean_abs_shap", "mean_shap", "shap_std", "shap_cv", "direction",
           "log2FC", "pvalue", "padj", "hedges_g_meta", "I2", "same_direction_datasets", "is_deg",
           "single_gene_auc", "single_gene_ci_lo", "single_gene_ci_hi", "in_overlap_panel", "in_top5_shap",
           "in_optimal_panel", "in_top5_deg", "gse7621_rank", "gse7621_importance", "gse7621_probe",
           "n_enriched_terms"]]
save_csv(ann, "13_gene_annotation_master.csv",
         "Master per-gene table: symbol, SHAP, DE stats, panel membership, AUC, external importance")
show_table(ann[["shap_rank", "symbol", "mean_abs_shap", "log2FC", "pvalue", "same_direction_datasets",
                "single_gene_auc", "in_optimal_panel", "gse7621_rank"]], "Master gene table:")

## 14. Robustness appendix

The original appendix quantified three reproduction quirks that do not exist
here. For the merged cohort the questions that matter are different. **Nothing
here overwrites the main results.**

| Variant | Question it answers |
|---|---|
| **A. Leave one dataset out** | Does a panel chosen on three datasets classify people in the fourth? Everything - DE, Boruta, SHAP, the best combination - is redone inside the three. |
| **B. Standard Boruta** | What does the original `perc=100` setting select on the same people? |
| **C. Nested CV panel selection** | What is the unbiased AUC of "pick the best five-gene panel"? |

In [ ]:
# ============================== 14A. LEAVE ONE DATASET OUT ==============================
robust_rows = []

def boruta_select(idx, perc=BORUTA_PERC, max_iter=BORUTA_MAX_ITER):
    sel = BorutaPy(RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_split=4,
                                          min_samples_leaf=2, class_weight="balanced", n_jobs=-1,
                                          random_state=RANDOM_STATE),
                   n_estimators="auto", perc=perc, alpha=BORUTA_ALPHA, two_step=True,
                   max_iter=max_iter, random_state=RANDOM_STATE, verbose=0).fit(X[idx], y[idx])
    cols = np.where(sel.support_)[0]
    if len(cols) < PANEL_SIZE:
        cols = np.argsort(sel.ranking_)[:PANEL_SIZE]
    return cols

def shap_top5(idx, cols, runs=3):
    acc = np.zeros(len(cols))
    for r_ in range(runs):
        m_ = RandomForestClassifier(n_estimators=800, max_depth=10, min_samples_split=4, min_samples_leaf=2,
                                    class_weight="balanced", random_state=RANDOM_STATE + r_ * 11,
                                    n_jobs=-1).fit(X[np.ix_(idx, cols)], y[idx])
        sv_ = shap.TreeExplainer(m_).shap_values(X[np.ix_(idx, cols)])
        sv_ = sv_[1] if isinstance(sv_, list) else (sv_[:, :, 1] if np.ndim(sv_) == 3 else sv_)
        acc += np.abs(sv_).mean(0)
    return cols[np.argsort(-acc)][:PANEL_SIZE]

def inner_cv_auc(idx, cols, seed=RANDOM_STATE):
    p = np.zeros(len(idx))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(idx, y[idx]):
        sc_ = StandardScaler().fit(X[np.ix_(idx[tr], cols)])
        lr = LogisticRegression(C=1.0, max_iter=1000).fit(sc_.transform(X[np.ix_(idx[tr], cols)]), y[idx[tr]])
        p[te] = lr.predict_proba(sc_.transform(X[np.ix_(idx[te], cols)]))[:, 1]
    return roc_auc_score(y[idx], p)

from joblib import Parallel, delayed
def best_combo_cv(idx, pool):
    combos = [list(c) for c in combinations(pool, PANEL_SIZE)]
    n_w = max(1, min(4, os.cpu_count() or 1))
    chunks = [combos[i::n_w] for i in range(n_w)]
    res = Parallel(n_jobs=n_w)(delayed(lambda cs: [(inner_cv_auc(idx, c), c) for c in cs])(ch)
                               for ch in chunks)
    s_, c_ = max((r_ for ch in res for r_ in ch), key=lambda t: t[0])
    return c_, s_

def fit_predict(tr, te, cols):
    sc_ = StandardScaler().fit(X[np.ix_(tr, cols)])
    lr = LogisticRegression(C=1.0, max_iter=1000).fit(sc_.transform(X[np.ix_(tr, cols)]), y[tr])
    return lr.predict_proba(sc_.transform(X[np.ix_(te, cols)]))[:, 1]

if RUN_ROBUSTNESS and RUN_LODO:
    print("=" * 62); print("VARIANT A - leave one dataset out"); print("=" * 62)
    LODO_MODELS = [f"Boruta panel", "Top-5 SHAP", "Top-5 DEG", "Best 5 (inner CV)", "All genes (RF)"]
    OOF = {k: np.full(len(y), np.nan) for k in LODO_MODELS}
    folds = []
    for d in DATASETS:
        t0 = time.time()
        te = np.where(DS == d)[0]; tr = np.where(DS != d)[0]
        _, _, p_tr, _ = wilcoxon_de(tr)
        top5_de_tr = list(np.argsort(p_tr)[:PANEL_SIZE])
        cols_b = boruta_select(tr)
        cols_s = list(shap_top5(tr, cols_b))
        pool_b = list(cols_b[:20]) if len(cols_b) > 20 else list(cols_b)
        cols_best, inner = best_combo_cv(tr, pool_b)
        for name, cols in (("Boruta panel", list(cols_b)), ("Top-5 SHAP", cols_s),
                           ("Top-5 DEG", top5_de_tr), ("Best 5 (inner CV)", cols_best)):
            OOF[name][te] = fit_predict(tr, te, cols)
        rf_ = RandomForestClassifier(n_estimators=800, max_depth=10, class_weight="balanced", n_jobs=-1,
                                     random_state=RANDOM_STATE).fit(X[tr], y[tr])
        OOF["All genes (RF)"][te] = rf_.predict_proba(X[te])[:, 1]
        overlap_with_main = len(set(np.array(gene_names)[cols_b]) & boruta_set)
        folds.append({"held_out": d, "n_test": len(te), "n_train": len(tr), "boruta_genes": len(cols_b),
                      "shared_with_main_panel": overlap_with_main,
                      "panel_boruta": " | ".join(sym(gene_names[i]) for i in cols_b),
                      "panel_top5_shap": " | ".join(sym(gene_names[i]) for i in cols_s),
                      "panel_top5_deg": " | ".join(sym(gene_names[i]) for i in top5_de_tr),
                      "panel_best5": " | ".join(sym(gene_names[i]) for i in cols_best),
                      "best5_inner_auc": inner})
        print(f"  held out {d:9s}: Boruta {len(cols_b)} genes ({overlap_with_main} shared with the main panel) "
              f"[{time.time() - t0:.0f}s]")
    lodo_folds = pd.DataFrame(folds)
    perf = []
    for name, s_ in OOF.items():
        ok = ~np.isnan(s_)
        lo, hi = bootstrap_auc_ci(y[ok], s_[ok])
        row = {"model": name, "people": int(ok.sum()), "pooled_auc": roc_auc_score(y[ok], s_[ok]),
               "ci_lo": lo, "ci_hi": hi,
               "mwu_p": stats.mannwhitneyu(s_[ok & (y == 1)], s_[ok & (y == 0)], alternative="greater").pvalue}
        for d in DATASETS:
            m_ = ok & (DS == d)
            row[f"auc_{d}"] = roc_auc_score(y[m_], s_[m_]) if len(set(y[m_])) == 2 else np.nan
        perf.append(row)
    lodo_perf = pd.DataFrame(perf)
    save_csv(lodo_folds, "14a_lodo_folds.csv", "Leave-one-dataset-out: what each fold selected")
    save_csv(lodo_perf, "14a_lodo_performance.csv", "Leave-one-dataset-out AUC, pooled and per held-out dataset")
    save_csv(pd.DataFrame({"person": META["person"], "dataset": DS, "y": y, **OOF}),
             "14a_lodo_predictions.csv", "Out-of-dataset prediction for every person")
    show_table(lodo_perf.round(3), "Leave-one-dataset-out performance:")
    for _, r_ in lodo_perf.iterrows():
        robust_rows.append({"variant": "A_leave_one_dataset_out", "metric": f"{r_['model']} pooled AUC",
                            "value": r_["pooled_auc"]})
else:
    print("Leave-one-dataset-out skipped.")

In [ ]:
# ============================== 14B. STANDARD BORUTA ==============================
if RUN_ROBUSTNESS:
    print("=" * 62); print("VARIANT B - Boruta with the original perc=100, max_iter=50"); print("=" * 62)
    std_sel = BorutaPy(RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_split=4,
                                              min_samples_leaf=2, class_weight="balanced", n_jobs=-1,
                                              random_state=RANDOM_STATE),
                       n_estimators="auto", perc=100, alpha=0.05, two_step=True, max_iter=50,
                       random_state=RANDOM_STATE, verbose=0).fit(X_train, y_train)
    std_genes = [gene_names[i] for i in np.where(std_sel.support_)[0]]
    std_tent  = [gene_names[i] for i in np.where(std_sel.support_weak_)[0]]
    shared = [g for g in std_genes if g in boruta_set]
    auc_std = panel_roc(std_genes, X_tr_full, X_te, y_tr_full, y_te)["auc"] if std_genes else np.nan
    print(f"Standard Boruta: {len(std_genes)} confirmed, {len(std_tent)} tentative; "
          f"{len(shared)}/{len(std_genes)} inside the broadened panel")
    print("  " + ", ".join(sym(g) for g in std_genes))
    print(f"Test AUC of the standard panel: {auc_std:.3f}  vs broadened {b20_roc['auc']:.3f}")
    save_csv(pd.DataFrame({"gene": std_genes + std_tent, "symbol": [sym(g) for g in std_genes + std_tent],
                           "status": ["confirmed"] * len(std_genes) + ["tentative"] * len(std_tent),
                           "in_broadened_panel": [g in boruta_set for g in std_genes + std_tent]}),
             "14b_boruta_standard_genes.csv", "Genes chosen by Boruta with the original perc=100, max_iter=50")
    robust_rows += [
        {"variant": "B_standard_boruta", "metric": "n_confirmed", "value": len(std_genes)},
        {"variant": "B_standard_boruta", "metric": "n_tentative", "value": len(std_tent)},
        {"variant": "B_standard_boruta", "metric": "n_inside_broadened_panel", "value": len(shared)},
        {"variant": "B_standard_boruta", "metric": "test_auc", "value": auc_std},
        {"variant": "main", "metric": "broadened_panel_size", "value": N_POOL},
        {"variant": "main", "metric": "broadened_panel_test_auc", "value": b20_roc["auc"]},
    ]

In [ ]:
# ============================== 14C. NESTED-CV PANEL SELECTION ==============================
if RUN_ROBUSTNESS and RUN_NESTED_CV:
    print("=" * 62); print("VARIANT C - nested CV for the 'best five-gene panel' procedure"); print("=" * 62)
    X_pool = get_cols(X, gene_pool)
    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pool_combos = list(combinations(range(len(gene_pool)), PANEL_SIZE))
    fold_rows = []
    for fold, (tr_idx, te_idx) in enumerate(outer.split(X_pool, STRATA), 1):
        X_in, X_out, y_in, y_out = X_pool[tr_idx], X_pool[te_idx], y[tr_idx], y[te_idx]
        Xi_tr, Xi_va, yi_tr, yi_va = train_test_split(X_in, y_in, test_size=0.25,
                                                      random_state=RANDOM_STATE, stratify=y_in)
        best_in, best_in_auc = None, -1.0
        for combo in pool_combos:
            cols = list(combo)
            sc_ = StandardScaler(); a = sc_.fit_transform(Xi_tr[:, cols]); b = sc_.transform(Xi_va[:, cols])
            lr = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE).fit(a, yi_tr)
            s_ = roc_auc_score(yi_va, lr.predict_proba(b)[:, 1])
            if s_ > best_in_auc:
                best_in_auc, best_in = s_, cols
        sc_ = StandardScaler(); a = sc_.fit_transform(X_in[:, best_in]); b = sc_.transform(X_out[:, best_in])
        lr = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE).fit(a, y_in)
        outer_auc = roc_auc_score(y_out, lr.predict_proba(b)[:, 1])
        genes_sel = [gene_pool[i] for i in best_in]
        fold_rows.append({"fold": fold, "inner_selection_auc": best_in_auc, "outer_test_auc": outer_auc,
                          "selected_panel": " | ".join(genes_sel),
                          "selected_symbols": " | ".join(sym(g) for g in genes_sel)})
        print(f"  fold {fold}: inner {best_in_auc:.3f} -> outer {outer_auc:.3f}   {', '.join(sym(g) for g in genes_sel)}")
    nested = pd.DataFrame(fold_rows)
    save_csv(nested, "14c_nested_cv_panel_selection.csv",
             "Nested CV: panel chosen on inner data, scored on the untouched outer fold")
    mean_outer = nested["outer_test_auc"].mean()
    reselected = sum(set(r_.split(" | ")) == set(best_genes) for r_ in nested["selected_panel"])
    print(f"\nNested-CV AUC : {mean_outer:.3f} +/- {nested['outer_test_auc'].std():.3f}")
    print(f"Test-set optimum: {best_auc:.3f}   optimism {best_auc - mean_outer:+.3f}")
    robust_rows += [
        {"variant": "C_nested_cv", "metric": "nested_cv_auc_mean", "value": mean_outer},
        {"variant": "C_nested_cv", "metric": "nested_cv_auc_std", "value": nested["outer_test_auc"].std()},
        {"variant": "C_nested_cv", "metric": "selection_optimism", "value": best_auc - mean_outer},
        {"variant": "C_nested_cv", "metric": "folds_reselecting_optimal_panel", "value": reselected},
        {"variant": "main", "metric": "optimal_panel_auc", "value": best_auc},
    ]

if RUN_ROBUSTNESS and robust_rows:
    save_csv(pd.DataFrame(robust_rows), "14_robustness_summary.csv", "All robustness variants side by side")
    show_table(pd.DataFrame(robust_rows), "Robustness variants:")

## 15. Figure-ready exports

ROC coordinates, per-person SHAP values with matching expression, the panel's
expression across all 63 people, and the Ensembl-to-symbol map for every gene,
so the figures notebook never re-runs the analysis.

In [ ]:
# ============================== FIGURE-READY EXPORTS ==============================
roc_rows = []
def _add_curve(name, kind, fpr, tpr, auc_val, ci_lo=np.nan, ci_hi=np.nan):
    for f, t in zip(fpr, tpr):
        roc_rows.append({"curve": name, "curve_type": kind, "fpr": f, "tpr": t,
                         "auc": auc_val, "ci_lo": ci_lo, "ci_hi": ci_hi})
_add_curve("Optimal (exhaustive)", "panel", best["fpr"], best["tpr"], best_auc, ci_lo_best, ci_hi_best)
for nm, r in [("Overlap (Boruta n DEG)", ov_roc), ("Top-5 SHAP", t5_roc), ("Top-5 DEG", deg5_roc),
              ("Boruta 20", b20_roc)]:
    if r is not None:
        _add_curve(nm, "panel", r["fpr"], r["tpr"], r["auc"], r["ci_lo"], r["ci_hi"])
for g in gene_pool:
    r = gene_roc[g]
    _add_curve(str(g), "single_gene", r["fpr"], r["tpr"], r["auc"], r["ci_lo"], r["ci_hi"])
_add_curve("GSE7621 transfer", "external", fpr_t, tpr_t, bt["auc"], bt["ci_lo"], bt["ci_hi"])
_add_curve("GSE7621 within-cohort", "external_within", fpr_e, tpr_e, auc_ext, ci_ext[0], ci_ext[1])
save_csv(pd.DataFrame(roc_rows), "15_roc_curves.csv",
         "ROC coordinates for every panel, single gene and the external cohort")

shap_rows = []
test_ds = DS[idx_test]
for j, g in enumerate(map(str, selected_genes)):
    for i in range(mean_shap.shape[0]):
        shap_rows.append({"gene": g, "sample_index": i, "shap_value": float(mean_shap[i, j]),
                          "feature_value": float(X_test_boruta[i, j]), "true_label": int(y_te[i]),
                          "dataset": test_ds[i]})
save_csv(pd.DataFrame(shap_rows), "15_shap_values_test.csv",
         "Per-person SHAP value and expression for each Boruta gene (test people)")

panel_expr = pd.DataFrame(get_cols(X, gene_pool), columns=gene_pool)
panel_expr.insert(0, "dataset", DS)
panel_expr.insert(0, "group", np.where(y == 1, "PD", "Control"))
panel_expr.insert(0, "sample_index", range(len(panel_expr)))
save_csv(panel_expr, "15_expression_panel_genes.csv",
         "Harmonised (z-scored) expression of the Boruta genes across all people, with group and dataset")
save_csv(pd.DataFrame({"gene": GENES, "symbol": [SYMBOL.get(g, "") for g in GENES]}),
         "15_gene_symbol_map.csv", "Ensembl ID to gene symbol for every harmonised gene")
print("\nFigure-ready exports complete.")

## 16. SHAP over every gene, not just the selected ones

SHAP on the baseline model over **all harmonised genes**, same 10-run protocol,
so Boruta's decision plays no part in the ordering.

In [ ]:
# ============================== GLOBAL SHAP ==============================
N_GLOBAL_SHAP_RUNS = 10
g_acc = np.zeros(X_train.shape[1]); g_sgn = np.zeros(X_train.shape[1])
for _i in range(N_GLOBAL_SHAP_RUNS):
    _m = RandomForestClassifier(n_estimators=800, max_depth=10, min_samples_split=4, min_samples_leaf=2,
                                class_weight="balanced", random_state=RANDOM_STATE + _i * 11, n_jobs=-1)
    _m.fit(X_train, y_train)
    _sv = shap.TreeExplainer(_m).shap_values(X_test)
    if isinstance(_sv, list):
        _sv = _sv[1]
    elif getattr(_sv, "ndim", 2) == 3:
        _sv = _sv[:, :, 1]
    g_acc += np.abs(_sv).mean(axis=0); g_sgn += _sv.mean(axis=0)
print(f"{N_GLOBAL_SHAP_RUNS} global SHAP runs over {X_train.shape[1]:,} genes done")
gshap = pd.DataFrame({"gene": list(gene_names), "symbol": [sym(g) for g in gene_names],
                      "mean_abs_shap": g_acc / N_GLOBAL_SHAP_RUNS, "mean_shap": g_sgn / N_GLOBAL_SHAP_RUNS})
gshap = gshap.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
gshap["rank"] = np.arange(1, len(gshap) + 1)
N_GENES = len(gshap)
gshap["in_boruta"] = gshap["gene"].isin(boruta_set)
n_hit20, n_hit50, n_hit100 = (int(gshap.head(k)["in_boruta"].sum()) for k in (20, 50, 100))
save_csv(gshap.head(200)[["rank", "gene", "symbol", "mean_abs_shap", "mean_shap", "in_boruta"]],
         "16_global_shap_top200.csv", "Top 200 genes by mean |SHAP| computed across all harmonised genes")
brank = gshap[gshap["in_boruta"]][["gene", "symbol", "rank", "mean_abs_shap"]].copy()
brank["percentile"] = 100.0 * (1.0 - (brank["rank"] - 1) / N_GENES)
brank = brank.sort_values("rank").reset_index(drop=True)
save_csv(brank, "16_boruta_rank_in_global_shap.csv", "Where each Boruta gene ranks in the all-gene SHAP ordering")
save_csv(pd.DataFrame([
    ("n_genes_ranked", N_GENES), ("n_runs", N_GLOBAL_SHAP_RUNS), ("n_boruta", len(boruta_set)),
    ("n_boruta_in_global_top20", n_hit20), ("n_boruta_in_global_top50", n_hit50),
    ("n_boruta_in_global_top100", n_hit100), ("boruta_median_rank", float(brank["rank"].median())),
    ("boruta_median_percentile", float(brank["percentile"].median())),
    ("expected_top20_overlap_by_chance", 20 * len(boruta_set) / N_GENES),
], columns=["statistic", "value"]), "16_global_shap_summary.csv",
   "Agreement between the all-gene SHAP ranking and the Boruta selection")
print(f"Of the global top 20 by SHAP, {n_hit20} are Boruta-selected "
      f"({20 * len(boruta_set) / N_GENES:.3f} expected by chance); median Boruta rank {brank['rank'].median():.0f}")

## 17. Enrichment of the panel itself, with the collection repaired

Hypergeometric test against the harmonised background, Benjamini-Hochberg, using
GO, KEGG, Reactome and Hallmark terms that hold **10 to 500 background genes**,
scored for the whole panel, the genes up in PD and the genes down.

In [ ]:
# ============================== PANEL ORA, REPAIRED ==============================
from scipy.stats import hypergeom as _hgm
MIN_TERM, MAX_TERM, MIN_HITS = 10, 500, 2
ORA_LIBS = ["GO_Biological_Process_2023", "GO_Cellular_Component_2023", "GO_Molecular_Function_2023",
            "KEGG_2021_Human", "Reactome_2022", "MSigDB_Hallmark_2020"]
BG_SYMS = {s.upper() for s in SYMBOL.values() if s}
N_BG = len(BG_SYMS)
print(f"Background: {len(GENES):,} genes -> {N_BG:,} symbols")
ORA_SETS = {}
for lib in ORA_LIBS:
    txt = None
    for k in range(4):
        try:
            with urllib.request.urlopen("https://maayanlab.cloud/Enrichr/geneSetLibrary"
                                        f"?mode=text&libraryName={lib}", timeout=300) as fh:
                txt = fh.read().decode("utf-8", "replace")
            break
        except Exception as exc:
            print(f"  {lib}: retry {k + 1} ({type(exc).__name__})"); time.sleep(10)
    if txt is None:
        continue
    n = 0
    for line in txt.rstrip("\n").split("\n"):
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        g = {x.split(",")[0].strip().upper() for x in parts[2:] if x.strip()} & BG_SYMS
        if MIN_TERM <= len(g) <= MAX_TERM:
            ORA_SETS[(lib, parts[0].strip())] = g; n += 1
    print(f"  {lib}: {n} terms")
print(f"{len(ORA_SETS):,} terms testable")

ann_sym = ann.copy(); ann_sym["sym"] = ann_sym["symbol"].fillna("").str.upper()
QUERIES = [("whole panel", set(ann_sym["sym"]) & BG_SYMS),
           ("up in PD", set(ann_sym.loc[ann_sym["log2FC"] > 0, "sym"]) & BG_SYMS),
           ("down in PD", set(ann_sym.loc[ann_sym["log2FC"] <= 0, "sym"]) & BG_SYMS)]

def run_ora(query, label):
    q = len(query); rows_ = []
    for (lib, name), g in ORA_SETS.items():
        ov = query & g
        rows_.append({"query": label, "library": lib, "term": name, "term_size": len(g), "overlap": len(ov),
                      "genes": ", ".join(sorted(ov)),
                      "pvalue": float(_hgm.sf(len(ov) - 1, N_BG, len(g), q)) if ov else 1.0})
    d = pd.DataFrame(rows_).sort_values("pvalue").reset_index(drop=True)
    m = len(d)
    d["FDR"] = np.minimum.accumulate((d["pvalue"] * m / (np.arange(m) + 1)).to_numpy()[::-1])[::-1].clip(0, 1)
    d["query_size"] = q
    return d

ora_all = pd.concat([run_ora(qs, lbl) for lbl, qs in QUERIES], ignore_index=True)
save_csv(ora_all[ora_all["overlap"] >= 1].sort_values(["query", "pvalue"]), "17_panel_ora_full.csv",
         "Panel over-representation against a size-filtered collection, by direction")
summary = []
for lbl, qs in QUERIES:
    d = ora_all[ora_all["query"] == lbl]
    sig = d[(d["FDR"] < 0.05) & (d["overlap"] >= MIN_HITS)]
    summary.append({"query": lbl, "query_size": len(qs), "terms_tested": len(d), "n_fdr05_min2genes": len(sig),
                    "best_FDR": float(d["FDR"].min()) if len(d) else np.nan,
                    "best_p": float(d["pvalue"].min()) if len(d) else np.nan})
    print(f"{lbl} (q={len(qs)}): {len(sig)} terms at FDR<0.05 with >={MIN_HITS} genes")
save_csv(pd.DataFrame(summary), "17_panel_ora_summary.csv", "How many terms survive per direction")
top = ora_all[(ora_all["query"] == "whole panel") & (ora_all["overlap"] >= MIN_HITS)].nsmallest(10, "pvalue")
drv = {}
for _, r in top.iterrows():
    for g in str(r["genes"]).split(", "):
        drv[g] = drv.get(g, 0) + 1
drivers = (pd.DataFrame({"symbol": list(drv), "n_top_terms": list(drv.values())})
           .sort_values("n_top_terms", ascending=False).reset_index(drop=True))
save_csv(drivers, "17_panel_ora_drivers.csv", "Which panel genes carry the strongest terms")
p_needed = 0.05 / max(len(ORA_SETS), 1)
save_csv(pd.DataFrame([
    ("background_symbols", N_BG), ("terms_tested", len(ORA_SETS)), ("min_term_size", MIN_TERM),
    ("max_term_size", MAX_TERM), ("min_overlap_required", MIN_HITS), ("p_needed_for_fdr05_at_rank1", p_needed),
], columns=["statistic", "value"]), "17_panel_ora_settings.csv", "Settings and the raw p needed at rank 1")

## 18. Expression matrix export (for permutation testing)

In [ ]:
# ============================== EXPRESSION EXPORT ==============================
npz = OUT_DIR / "18_hvg_expression.npz"
np.savez_compressed(npz, X=X.astype(np.float32), y=y.astype(np.int8),
                    genes=np.array(GENES, dtype="U24"), dataset=np.array(DS, dtype="U12"),
                    person=np.array(META["person"], dtype="U32"))
WRITTEN.append({"file": "18_hvg_expression.npz", "rows": X.shape[0], "columns": X.shape[1],
                "description": "Harmonised z-scored matrix with PD/Control labels and dataset, for permutation testing"})
print(f"wrote {npz.name}: {X.shape[0]} people x {X.shape[1]} genes")

## 19. Output manifest

In [ ]:
# ============================== MANIFEST ==============================
manifest = pd.DataFrame(WRITTEN).drop_duplicates(subset="file").reset_index(drop=True)
manifest.insert(0, "n", range(1, len(manifest) + 1))
manifest.to_csv(OUT_DIR / "00_manifest.csv", index=False)
print(f"{len(manifest)} files written to {OUT_DIR.resolve()}\n")
show_table(manifest[["n", "file", "rows", "description"]], "Files written:")

print("\n" + "=" * 66)
print("HEADLINE RESULTS")
print("=" * 66)
print(f"  Cohort                 : {X.shape[0]} people x {X.shape[1]:,} genes, {len(DATASETS)} datasets")
print(f"  Dataset identity       : {acc_before:.2f} before -> {acc_after:.2f} after harmonisation (chance {chance:.2f})")
print(f"  Baseline RF accuracy   : {test_acc:.3f} (test n = {len(idx_test)})")
print(f"  DE, FDR < 0.05         : {int((de_results['padj'] < DE_FDR).sum())}   (set used: {DE_LABEL}, n = {len(degs)})")
print(f"  Boruta genes           : {N_POOL}   (perc {BORUTA_PERC})")
print(f"  Boruta n DE overlap    : {len(overlap)}")
if ov_roc is not None:
    print(f"  Overlap panel AUC      : {ov_roc['auc']:.3f} [{ov_roc['ci_lo']:.3f}-{ov_roc['ci_hi']:.3f}]")
print(f"  Boruta panel AUC       : {b20_roc['auc']:.3f} [{b20_roc['ci_lo']:.3f}-{b20_roc['ci_hi']:.3f}]")
print(f"  Top-5 SHAP panel AUC   : {t5_roc['auc']:.3f}")
print(f"  Top-5 DE panel AUC     : {deg5_roc['auc']:.3f}")
print(f"  Optimal panel AUC      : {best_auc:.3f} (test-set selection, {len(combo_results):,} panels)")
print(f"  GSE7621 transfer AUC   : {bt['auc']:.3f} [{bt['ci_lo']:.3f}-{bt['ci_hi']:.3f}], p = {bt['perm_p']:.4f}")
print(f"     after neuron adjust : {ba['auc_after_neuron_adjustment']:.3f}")
print(f"  GSE7621 within-cohort  : {auc_ext:.3f} hold-out | {cv_ext.mean():.3f} repeated CV")
if RUN_ROBUSTNESS and RUN_LODO:
    _b = lodo_perf.set_index("model").loc["Boruta panel"]
    print(f"  Leave-one-dataset-out  : Boruta panel {_b['pooled_auc']:.3f} [{_b['ci_lo']:.3f}-{_b['ci_hi']:.3f}]")
if RUN_ROBUSTNESS and RUN_NESTED_CV:
    print(f"  Nested-CV panel AUC    : {mean_outer:.3f} (optimism {best_auc - mean_outer:+.3f})")
print("=" * 66)